In [ ]:
#Import necessary libraries

#%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.table import Table
import scipy.stats
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.coordinates import Galactic
from astropy.coordinates import ICRS
import astropy.coordinates as apycord
import random
from cycler import cycler
import astropy.table

import re
import csv
import sys
import scipy
import pickle
import astropy
import pylab as p
import random as random
import numpy.random as rand
import astropy.io.fits as pyfits


from math import *
from math import cos, sin, pi
from numpy import *
from matplotlib import *
from scipy.interpolate import *
from astropy.table import join
from matplotlib.colors import LogNorm
import statistics as stat

# from scipy.integrate import trapz
# from scipy.integrate import quad

from astropy.table import QTable
from astropy.table import Table
import astropy.units as u
from astropy.coordinates import Angle

import zipfile as zf
import glob
from astropy.io import fits
import tarfile
import gzip
import shutil
import pandas as pd
from scipy.optimize import curve_fit

from concurrent.futures import ThreadPoolExecutor

import emcee
import corner

from PIL import Image
from IPython.display import display
import os

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', 'black']
zs = np.arange(2.5,7.5)
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0 = 70, Om0 =  0.3, Tcmb0 = 2.725)


In [ ]:
import time

In [ ]:
#Read in the simulation grid produced by Pandeia in the previous notebook
pandeia_simgrid = Table.read("sim_grid.fits")

In [ ]:
#Display one example entry from the Pandeia simulation grid
pandeia_simgrid[213]

In [ ]:
#Iterating through noise variations for each element in the redshift*luminosity bin grid
#Randomizing the seed using time.time so that Pandeia doesn't auto reset it to the same randomization every time
noisy_fluxes_bright=[]
noisy_fluxes_dim=[]
np.random.seed(int(time.time()*100)%123456789)
for i in range(5):
    

        #Generating noise from the calibrated extracted_noise
    noise_arr = np.random.normal(0, pandeia_simgrid['Noise Arrays'][213])
    noisy_fluxes_bright.append(pandeia_simgrid['Flux Arrays'][213] + noise_arr)
    
for i in range(5):

        #Generating noise from the calibrated extracted_noise
    noise_arr = np.random.normal(0, pandeia_simgrid['Noise Arrays'][187])
    noisy_fluxes_dim.append(pandeia_simgrid['Flux Arrays'][187] + noise_arr)

In [ ]:
#Testing that there are 5 different noise iterations randomly generated
print([i[0] for i in noisy_fluxes_bright])

# Now, I will be creating an EMCEE pipeline for my simulated spectra

In [ ]:
#Calculates H-alpha observed wavelength from redshifts
h_alphas = [6563*(z+1) for z in pandeia_simgrid['Redshifts']]

In [ ]:
#creates wavelength, flux, and flux error arrays from the Pandeia simulation grid
a=[]
b=[]
c=[]
for i in range(len(pandeia_simgrid)):
    a.append(pandeia_simgrid['Wavelength Arrays'][i])
    b.append(pandeia_simgrid['Flux Arrays'][i])
    c.append(pandeia_simgrid['Noise Arrays'][i])


In [ ]:
#Rename variables appropriately
waves = a
fluxes = b
flux_errs = c

In [ ]:
#Defining a gaussian curve fit
def gauss(x, A, mu, sigma):
    return A*np.exp(-((x-mu)**2)/(2*(sigma**2)))

#Defining a flat continuum fit
def flat(x,b):
    return np.ones(len(x))*b

#Defining a linear continuum fit
def lin(x,m,b):
    return (m*x) + b

#Combining a Gaussian with a flat continuum
def comb(x,A, mu, sigma, b):
    return gauss(x,A, mu, sigma) + flat(x,b)

#Combining a Gaussian with a linear continuum
def comb2(x,A,mu,sigma,m,b):
    return gauss(x,A,mu,sigma) + lin(x,m,b)

In [ ]:
#Define the likelihood function that EMCEE uses to evaluate parameter jumps
def lnlike(theta, x, y, yerr):
    model = comb(x, *theta)
    lnL = -0.5*np.sum((y - model) ** 2 / yerr ** 2)
    return lnL

#Set constraints for EMCEE parameter values, returning negative infinity if a value doesn't fit these priors
def lnprior(theta, wave_center, Amp_max):
    A, mu, sigma, b = theta #from the Gaussian comb fit
    
    #setting priors
    left_mu = wave_center - .25#e4 #double check, is this in microns or in pixels, pretty sure pixel window for consistency
    right_mu = wave_center + .25#e4
    min_A = 0.5*Amp_max 
    max_A = 2*Amp_max
    sigma_left = 0
    sigma_right = 0.15#e4
    
    #applying priors
    if (min_A < A < max_A) & (left_mu <= mu <= right_mu) & (sigma_left < sigma < sigma_right) & (b>0):
        return 0.0
    return -np.inf

#combines the lnlike and lnprior functions together, determining whether or not a proper value is returned for an EMCEE paramter jump
def lnprob(theta, x, y, yerr, first_wave, Amp_max):
    lp = lnprior(theta, first_wave, Amp_max)
    if not np.isfinite(lp):
        return -np.inf
    prob = lp + lnlike(theta, x, y, yerr)
    return prob


#Make an initial guess for the parameters fitting H-alpha (amplitude, center, width) using curve_fit
def init_emcee(wave, spectrum, err_spec, window, line_center, diagnose = False): #diagnose will run tests
    #Wave, spectrum, and err_spec are all the full arrays, not zoomed  
    
#     window_check = np.log10(window)
    
#      if window_check > -2:
#         print('WARNING: Search window may be too big double check input value')
    min_window = line_center - window 
    max_window = line_center + window
    
    
    indx = np.where((min_window < wave) & ((wave < max_window)))[0] #Gives indexes of zoomed waves/fluxes
    
    spec_window = spectrum[indx] #Applies index mask to flux, wavelengths, and flux_errs
    wave_window = wave[indx]
    err_spec_window = err_spec[indx]
    
    
    guess_A = np.amax(spectrum[indx]) #The highest flux value is the amp init guess
    guess_mu = line_center #The line center is the mu init guess
    
    #Fancy interpolation along all the points in the zoomed window
    spec_interp = Akima1DInterpolator(wave_window, spec_window)
    x = np.linspace(wave_window[0], wave_window[-1], 10000)
    spec = spec_interp(x)
    
    half_max = np.amax(spec)/2 #Gives the half_max flux amp
    
    #To guess sigma, find index where flux is above half_max
    idx = np.where(spec > half_max)[0] #gives indices that satisfy above condition
    wave_left, wave_right = x[idx[0]], x[idx[-1]] #first and last idx indices are where sigma is calculated
    guess_sigma = (wave_right - wave_left)/2
    
    if diagnose == True:
        
        print('Minimization Guesses')
        print(f"A: {guess_A}")
        print(f"mu: {guess_mu}")
        print(f"sigma: {guess_sigma}")
        print(f"b: {np.median(spec_window)}")
        print() 

    x0 = [guess_A*1e20, guess_mu, guess_sigma, np.median(spec_window*1e20)] #Initial guesses made
    bounds_low = [0, 0, 0, -np.inf] #maybe adjust later
    bounds_high = [np.inf, np.inf, np.inf, np.inf]
    
    result,_ = curve_fit(comb, wave_window, spec_window*1e20, p0 = x0, bounds = [bounds_low, bounds_high])  
    result[0]/=1e20
    result[3]/=1e20
    
     ########
    # Diagnostic Plotting: making sure we are getting the emission line
    ########
    if diagnose == True:
        
        print('Minimization Results')
        print(f"A: {result[0]}")
        print(f"mu: {result[1]}")
        print(f"sigma: {result[2]}")
        print(f"b: {result[3]}")
        print()
        
        xarr = np.linspace(wave_window[0], wave_window[-1], 100)
#         plt.figure()
#         plt.plot(wave_window, spec_window, color = 'blue', label = 'Data')
#         plt.scatter(wave_window, spec_window, color = 'blue')
#         plt.plot(xarr, comb(xarr, *result), color = 'black', label = 'Model')
#         plt.axhline(0, linestyle = '--')
#         plt.ylabel('Flux')
#         plt.xlabel(r'Wavelength $\mu$m')
#         plt.title('Initial curve_fit Fitting')
#         plt.legend()
#   #      plt.show()
    
    return result 


#EMCEE actually creates walkers, and jumps through the parameter space starting from the initial parameter guesses
def fitting_line(wave, flux, flux_err, line_center, window_wavelength, run = 3000,
                 diagnose = False,save_df=True, save_spec = True, 
                 file_spec = 'Emcee_Spectra_Test.txt', 
                 filename = 'Emcee_Chains_Galaxy_Test.txt'):
    
    result = init_emcee(wave, flux, flux_err, window_wavelength, line_center, diagnose = diagnose) #calls from above
    print(result)
    guess_A = result[0]
    guess_mu = result[1]
    guess_sigma = result[2]
    guess_b = result[3]
    
    #Now, create walkers to explore the parameter space
    amp_jump = np.random.normal(loc = guess_A,        
                                scale = guess_A/10,      
                                size = 32).reshape(-1, 1) 
    
    wavelength_jump = np.random.normal(loc = guess_mu,    
                                       scale = .005,      #this is in microns
                                       size = 32).reshape(-1, 1)
    
    sigma_jump = np.random.normal(loc = guess_sigma,       
                                  scale = .002,           #also in microns
                                  size = 32).reshape(-1, 1)
    powerb = np.log10(np.abs(guess_b))
    b_jump = np.random.normal(loc = guess_b,           #centered on best b from curve_fit
                              scale = 1*10**powerb,    #making it wander 10^powerb (if b = .05, it can wander .01)
                              size = 32).reshape(-1, 1)
    
    #################
    # Diagnostic plotting to see if the parameters were jumping to large values
    # The should be concentrated near their best fit results values
    #################
    if diagnose == True:
        print('Checking the Walker Jumps')
        fig, ax = plt.subplots(nrows = 2, ncols = 2, constrained_layout = True)
        
        ax[0, 0].hist(amp_jump)
        ax[0, 0].set_xlabel('Amplitude')
        
        ax[0, 1].hist(wavelength_jump)
        ax[0, 1].set_xlabel(r'$\mu$')
        
        ax[1, 0].hist(sigma_jump)
        ax[1, 0].set_xlabel(r'$\sigma$')
        
        ax[1, 1].hist(b_jump)
        ax[1, 1].set_xlabel('b')
        
   #     plt.show()
        
    #stacking along columns and creating starter walkers    
    starting_walkers = np.hstack((amp_jump, wavelength_jump, sigma_jump, b_jump))
    
    emcee_window = window_wavelength
    emcee_indx = np.where((wave >= (line_center - emcee_window)) & (wave <= (line_center + emcee_window)))[0] 
    
    emcee_spec = flux[emcee_indx]
    emcee_wave = wave[emcee_indx]
    emcee_err = flux_err[emcee_indx]
    
    #NOTE:
    #need to change output name everytime you run otherwise it will overwrite
    ###########
    
    if save_spec == True:
        #saves the input emcee spectra
        emcee_spec_matrix = np.c_[emcee_wave, emcee_spec, emcee_err]
    
        np.savetxt(file_spec, emcee_spec_matrix)
        
    #initializing walker positions
    pos = starting_walkers
    nwalkers, ndim = pos.shape
    
    #initializing sampler
    sampler = emcee.EnsembleSampler(nwalkers, #giving emcee the walker positions
                                    ndim,     #giving it the dimension of the model(same as number of model parameters)
                                    lnprob, #giving it the log_probability function
                                    args=(emcee_wave, emcee_spec, emcee_err, guess_mu, guess_A))#, #arguments to pass into log_probability
#                                     moves = [(emcee.moves.DEMove(), 0.5),        
#                                              (emcee.moves.DESnookerMove(), 0.5)])
    
     #running emcee
    state = sampler.run_mcmc(pos, 1000)
    sampler.reset()
    sampler.run_mcmc(state, run, progress=False)
    
    #getting values back
    flat_samples = sampler.get_chain(flat=True)
    print(flat_samples)
    LnL_chain = sampler.flatlnprobability
   # burn_in = 1000 
    
    emcee_df = pd.DataFrame()
    emcee_df['A'] = flat_samples[:, 0]
    emcee_df['mu'] = flat_samples[:, 1]
    emcee_df['sigma'] = flat_samples[:, 2]
    emcee_df['b'] = flat_samples[:, 3]
    emcee_df['LnL'] = LnL_chain[:]
  #  print(emcee_df)
    emcee_df = emcee_df[np.isfinite(emcee_df.LnL.values)] #removing bad log likelihood fits

    
    int_fluxes_emcee = (emcee_df['A']) * emcee_df['sigma'] * np.sqrt(2 * np.pi) #getting the flux from the parameter values
#    int_flux_errs_emcee = int_fluxes_emcee*np.sqrt((A_err/emcee_df['A'])**2 + (sigma_err/emcee_df['sigma'])**2) #getting the flux_errors from the parameter values
    
    emcee_df['Fluxes'] = int_fluxes_emcee
   # emcee_df['Flux Errors'] = int_flux_errs_emcee
    
   # maxprob = np.argmax(emcee_df['LnL'])
   # emcee_df.iloc(maxprob, :-1)
    
    
#     if diagnose == True:
        
#         print('Checking Parameter Posterior Distributions')
#         fig, ax = plt.subplots(nrows = 2, ncols = 2, constrained_layout = True)
        
#         emcee_df.A.hist(ax = ax[0, 0])
#         emcee_df.mu.hist(ax = ax[0, 1])
#         emcee_df.sigma.hist(ax = ax[1, 0])
#         #emcee_df.m.hist(ax = ax[1, 0])
#         emcee_df.b.hist(ax = ax[1, 1])
        
#   #      plt.show()
    
    if diagnose == True:
        xarr = np.linspace(emcee_wave[0], emcee_wave[-1], 100)
        
#         plt.figure()
#         plt.title('Input Emcee Spectra and Emcee Fit')
#         plt.plot(emcee_wave, emcee_spec, color = 'black', alpha = 0.5, label = 'Data')
#         plt.scatter(emcee_wave, emcee_spec, color = 'black')
#         plt.plot(xarr, comb(xarr, *emcee_df.quantile(q = 0.5).values[:-2]), label = 'Model')
#         plt.xlabel(r'Wavelength [$\mu$m]')
#         plt.ylabel('Flux')
#         plt.legend()
#   #      plt.show()
    
    
    
    
    xarr = np.linspace(emcee_wave[0], emcee_wave[-1], 100)

    line_center = guess_mu
    
#     plt.figure()
#     plt.title('Input Emcee Spectra and Emcee Fit WITH 1-SIGMA SHADING')
#     plt.step(emcee_wave, emcee_spec, color = 'red', alpha = 0.5, label = 'Data', where = 'mid')
#     plt.errorbar(emcee_wave, emcee_spec, yerr = emcee_err, color = 'gray', alpha = 0.5, fmt = 'none')
    
#     l16_params = emcee_df.quantile(q = 0.16).values[:-2]
#     u84_params = emcee_df.quantile(q = 0.84).values[:-2]

#     l16_model = comb(xarr, *l16_params)#, line_center )
#     med_model = comb(xarr, *emcee_df.quantile(q = 0.5).values[:-2])#, line_center )
#     u84_model = comb(xarr, *u84_params)#, line_center )
    
#     plt.plot(xarr, med_model, label = 'Model', color = 'black')
    
#     plt.fill_between(xarr, u84_model, l16_model, color = 'dodgerblue', alpha = 0.5)
    
#     plt.xlabel(r'Wavelength [$\mu$m]')
#     plt.ylabel('Flux')
#     plt.legend()
#   #  plt.show()
#     ###########
#     #NOTE:
#     #need to also give the filename argument otherwise it will overwrite the default file
#     ###########
    if save_df == True:
        emcee_df.to_csv(filename, sep = ' ')
        
    else:
        return emcee_wave, emcee_spec, emcee_err, emcee_df


    

In [ ]:
#Test EMCEE Gaussian curve fitting on one simulated spectra with all its noise iterations (5)
for grid in range(1):
    line_window = .3#e4

    #best_lamba_obs = waves[19][ha_centers[19]]
    best_lamba_obs = .6563*(1 + pandeia_simgrid['Redshifts'][213])

    check_wave, check_flux, check_flux_err, check_df = fitting_line(waves[213], 
                                                                    noisy_fluxes_bright[grid], flux_errs[213], 
                                                                    best_lamba_obs, line_window, save_df = False,
                                                                    diagnose = True)
    xarr = np.linspace(check_wave[0], check_wave[-1], 1000)

    l16_params = check_df.quantile(q = 0.16).values[:-2]
    u84_params = check_df.quantile(q = 0.84).values[:-2]

    l16_model = comb(xarr, *l16_params)#, line_center )
    med_model = comb(xarr, *check_df.quantile(q = 0.5).values[:-2])#, line_center )
    u84_model = comb(xarr, *u84_params)#, line_center 

    #Getting the median parameters from the emcee fit
    median_param_vals = check_df.quantile(q = 0.5).values[:-2] #We use the [:-2] because the last two columns 
                                                               #are LnL and Flux
                                                               #We do not need those for the line_model 

    print("Median Param Vals: ")
    print(median_param_vals)
    #max_likely_vals = check_df.iloc[check_df.LnL.argmax(),:-2].values #doing max_likelihood values too

    dot1 = median_param_vals[1] - 3*median_param_vals[2] #Gets the bounds of the wavelength 3-sigma range
    dot2 = median_param_vals[1]
    dot3 = median_param_vals[1] + 3*median_param_vals[2]
    print("dots")
    print(dot1)
    print(dot3)
    

    baseline = check_df.quantile(q=0.5)[3] #Baseline value to subtract from data flux points

    #sigma3wave=[] #find wavelength range within 3-sigma
    sigma3ind=[] #get the indices of this range

    for i in range(len(waves[213])): 
        if waves[213][i]> dot1 and waves[213][i] < dot3:
           # sigma3wave.append(waves[19][i])
            sigma3ind.append(i)
    check_flux, sigma3ind
    print("sigma3ind: ")
    print(sigma3ind)

    sumerrs = []
    for i in sigma3ind: #Loop through indices of 3-sigma range to get flux, subtract baseline
        sumerrs.append(flux_errs[213][i]) #collect errors within this range as well


    flux_bits = []
    for i in sigma3ind:
        flux_bits.append((noisy_fluxes_bright[grid][i])*(waves[213][i+1]-waves[213][i]))
    data_fluxes = sum(flux_bits)


    prop=[]
    for i in sumerrs:
        prop.append(i)
    prop_err = sum(prop)*(dot3-dot1)
#    fit_fluxes, data_fluxes, prop_err
    
    #SNR calculation:
    #median/(median-l16param)
    
    med_amp = check_df.quantile(q = 0.5)[0]
    low_amp = check_df.quantile(q = 0.16)[0]

    snr_peak = med_amp/(med_amp - low_amp)


    #print("Median: " + str(median_param_vals) + " Max_Likely: " + str(max_likely_vals)) #compare median and max likelihood
    print("Median Integrated Flux: " + str(check_df.quantile(q = 0.5).values[-1]))
    print("Flux from Data: " + str(data_fluxes) + " ± " + str(prop_err))
    print("Baseline: " + str(baseline))

    fig, ax = plt.figure(figsize = (18, 10))
    plt.step(check_wave, check_flux, color = 'cadetblue', label = 'Data', where ='mid')
    plt.scatter(check_wave, check_flux, color = 'cadetblue')
    plt.plot(xarr, comb(xarr, *median_param_vals), label = 'Model', color = "black")

    #plt.plot(xarr, comb(xarr, *max_likely_vals), label = 'Model', color = "red")
    plt.errorbar(check_wave, check_flux, yerr = check_flux_err, fmt = "none", capsize = 3, color = "cadetblue")
    plt.axvline(dot1, color = 'red', linestyle = '--', label = "3-sigma")
    #plt.axvline(dot2, color = 'red', linestyle = '--')
    plt.axvline(dot3, color = 'red', linestyle = '--')
    plt.fill_between(xarr, u84_model, l16_model, color = 'dodgerblue', alpha = 0.5)
    plt.xlabel("Wavelength (Microns)", fontsize = 35)
    plt.ylabel("Flux (Jansky)", fontsize = 35)
    #plt.ylabel("Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
    #plt.title("First EMCEE Fit for " + str(capstone_rubies['filename'][19]))
    plt.title("H-α Emission Line with MCMC Gaussian Fit", fontsize = 40)
    plt.legend(fontsize = 30)
    plt.gca().get_yaxis().get_offset_text().set_fontsize(24)
    plt.tick_params(axis = "both", labelsize = 20)
    fig.text(.87, .6, 'z = ' + str(pandeia_simgrid[213]['Redshifts']), ha='center', fontsize = 20)
    fig.text(.79,.55, 'Integrated Flux = ' + str(pandeia_simgrid[213]['Integrated Fluxes'])[0:5] + str(pandeia_simgrid[213]['Integrated Fluxes'])[-4:], ha = 'center', fontsize = 20)
    fig.text(.8, .5, 'SNR = ' + str(snr_peak)[:5], fontsize = 20)
    plt.show()
    

In [ ]:
#Plot the EMCEE fit for one noise iteration of a single simulated H-alpha emission line
fig  = plt.figure(figsize = (18, 10), dpi = 250)
plt.step(check_wave, check_flux, color = 'cadetblue', label = 'Data', where ='mid')
plt.scatter(check_wave, check_flux, color = 'cadetblue')
plt.plot(xarr, comb(xarr, *median_param_vals), label = 'Model', color = "black")

 #plt.plot(xarr, comb(xarr, *max_likely_vals), label = 'Model', color = "red")
plt.errorbar(check_wave, check_flux, yerr = check_flux_err, fmt = "none", capsize = 3, color = "cadetblue")
plt.axvline(dot1, color = 'red', linestyle = '--', label = "3-sigma")
#plt.axvline(dot2, color = 'red', linestyle = '--')
plt.axvline(dot3, color = 'red', linestyle = '--')
plt.fill_between(xarr, u84_model, l16_model, color = 'dodgerblue', alpha = 0.5)
plt.xlabel("Wavelength (Microns)", fontsize = 35)
plt.ylabel("Flux (Jansky)", fontsize = 35)
#plt.ylabel("Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
#plt.title("First EMCEE Fit for " + str(capstone_rubies['filename'][19]))
plt.title("H-α Emission Line with MCMC Gaussian Fit", fontsize = 40)
plt.legend(fontsize = 30)
plt.gca().get_yaxis().get_offset_text().set_fontsize(20)
plt.tick_params(axis = "both", labelsize = 20)
fig.text(.87, .6, 'z = ' + str(pandeia_simgrid[213]['Redshifts']), ha='center', fontsize = 20)
fig.text(.79,.55, 'Integrated Flux = ' + str(pandeia_simgrid[213]['Integrated Fluxes'])[0:5] + str(pandeia_simgrid[213]['Integrated Fluxes'])[-4:], ha = 'center', fontsize = 20)
fig.text(.8, .5, 'SNR = ' + str(snr_peak)[:5], fontsize = 20)
plt.show()

#  Big EMCEE Loop

In [ ]:
#Loop through all the redshift and luminosity bins in by Pandeia simulation grid, and run 5 noise iterations on each one
#Fit EMCEE Gaussin curves to all of these emission lines, and calculate the SNR for each fit, as SNR<5 will be marked as not fit

snr_master =[]
count=0
for elem in range(len(pandeia_simgrid)): #iterate through each z+int_flux element in the 341-gridspace
    try:
        wave_grid = waves[elem]
        fluxes_grid = fluxes[elem]
        noises_grid = flux_errs[elem]

        noisy_fluxes=[]
        np.random.seed(int(time.time()*100)%123456789) #set random seed for 5 noise iterations

        for i in range(5):
            noise_arr = np.random.normal(0, flux_errs[elem])
            noisy_fluxes.append(fluxes[elem] + noise_arr)

        for noise in range(len(noisy_fluxes)):
            line_window = .3#e4

            #best_lamba_obs = waves[19][ha_centers[19]]
            best_lamba_obs = .6563*(1 + pandeia_simgrid['Redshifts'][elem])

            check_wave, check_flux, check_flux_err, check_df = fitting_line(waves[elem], 
                                                                            noisy_fluxes[noise], flux_errs[elem], 
                                                                            best_lamba_obs, line_window, save_df = False,
                                                                            diagnose = True)
            xarr = np.linspace(check_wave[0], check_wave[-1], 1000)

            l16_params = check_df.quantile(q = 0.16).values[:-2]
            u84_params = check_df.quantile(q = 0.84).values[:-2]

            l16_model = comb(xarr, *l16_params)#, line_center )
            med_model = comb(xarr, *check_df.quantile(q = 0.5).values[:-2])#, line_center )
            u84_model = comb(xarr, *u84_params)#, line_center )

            #Getting the median parameters from the emcee fit
            median_param_vals = check_df.quantile(q = 0.5).values[:-2] #We use the [:-2] because the last two columns 
                                                                       #are LnL and Flux
                                                                       #We do not need those for the line_model 
            print("Median Param Vals: ")
            print(median_param_vals)
            print(median_param_vals[1], median_param_vals[2])

            #max_likely_vals = check_df.iloc[check_df.LnL.argmax(),:-2].values #doing max_likelihood values too

            dot1 = median_param_vals[1] - 3*median_param_vals[2] #Gets the bounds of the wavelength 3-sigma range
            #dot2 = median_param_vals[1]
            dot3 = median_param_vals[1] + 3*median_param_vals[2]




            med_amp = check_df.quantile(q = 0.5)[0]
            low_amp = check_df.quantile(q = 0.16)[0]

            snr_peak = med_amp/(med_amp - low_amp)


            #print("Median: " + str(median_param_vals) + " Max_Likely: " + str(max_likely_vals)) #compare median and max likelihood
            print("Median Integrated Flux: " + str(check_df.quantile(q = 0.5).values[-1]))
            print("Flux from Data: " + str(data_fluxes) + " ± " + str(prop_err))
            print("Baseline: " + str(baseline))

            fig = plt.figure(figsize = (10, 7))
            plt.step(check_wave, check_flux, color = 'cadetblue', label = 'Data', where ='mid')
            plt.scatter(check_wave, check_flux, color = 'cadetblue')
            plt.plot(xarr, comb(xarr, *median_param_vals), label = 'Model', color = "black")

            #plt.plot(xarr, comb(xarr, *max_likely_vals), label = 'Model', color = "red")
            plt.errorbar(check_wave, check_flux, yerr = check_flux_err, fmt = "none", capsize = 3, color = "cadetblue")
            plt.axvline(dot1, color = 'red', linestyle = '--', label = "3-sigma")
            #plt.axvline(dot2, color = 'red', linestyle = '--')
            plt.axvline(dot3, color = 'red', linestyle = '--')
            plt.fill_between(xarr, u84_model, l16_model, color = 'dodgerblue', alpha = 0.5)
            plt.xlabel("Wavelength (Microns)")
            plt.ylabel("Flux (Jy)")
            #plt.ylabel("Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
            #plt.title("First EMCEE Fit for " + str(capstone_rubies['filename'][19]))
            plt.title("Pandeia EMCEE Fit Number 1")
            plt.legend()
            fig.text(.85, .7, 'z = ' + str(pandeia_simgrid[elem]['Redshifts']), ha='center')
            fig.text(.8,.65, 'Integrated Flux = ' + str(pandeia_simgrid[elem]['Integrated Fluxes'])[0:5] + str(pandeia_simgrid[elem]['Integrated Fluxes'])[-4:], ha = 'center')
            fig.text(.8, .6, 'SNR = ' + str(snr_peak)[:5])
            plt.show()
            snr_master.append(snr_peak)
            count+=1
            if count%20 ==0:
                print(count)
    except Exception as e:
        #err_ind.append(i)
        print("EMCEE Failed for z = " + str(pandeia_simgrid['Redshifts'][elem]) + ', int_flux = ' + str(pandeia_simgrid['Integrated Fluxes'][elem]) + ", Noise Iteration " + str(noisy_fluxes[noise]))
        print(e)
        snr_master.append(0)
        count+=1
        if count%20 == 0:
            print(count)

# Fix 16 missing runs

In [ ]:
#Each add in must be done sequentially. for example, 1 at 212. then rerun. then 2 at 316. and so on

#z = 2: 0-154 (155)
#z = 2.5: 155-309 (154) (1 missing in 1.258e-18 bin, add in index 212)
#z = 3: 309:459 (150) (2 missing in 1.258e-19, add in 2 at 316) (3 missing at 3.162e-19, add in 3 at 335)
#z = 3.5 459:613 (154) (1 missing in 3.162e-19, add in 1 at 485)
#z = 4 613:764 (151) (3 missing in 1e-19, 1 missing in 1.995e-19)
#z = 4.5 764:919 (155)
#z = 5 919:1074 (155)
#z = 5.5: 1074:1226 (152) (3 missing in 1.995e-19)
#z = 6: 1226:1381 (155)
#z = 6.5: 1381:1535 (154) (1 missing at 6.309e-19)
#z = 7: 1535:1689 (154) (1 missing at 1e-19)
#Missing 16 runs!

In [ ]:
#Correcting the issue of skipping noise iterations after EMCEE failed for 1 in that z+intflux bin

zero_inds = [i for i, x in enumerate(snr_master) if x == 0]
#print(zero_inds)  # Output: [1, 3, 5]
zero_inds


# Find what bins are 0% or 100% complete

In [ ]:
#replace SNR = nan with SNR = 0
nan_inds=[]
for i in range(len(snr_master)):
    if math.isnan(snr_master[i]):
        nan_inds.append(i)
        
for i in nan_inds:
    snr_master[i] = 0

In [ ]:
#Splitting up EMCEE results by redshift range
chunked = [snr_master[i:i + 5] for i in range(0, len(snr_master), 5)]

print(len(chunked))  # Output: 341
#print(chunked[:3])  # Print first 3 sublists to check
rerun_chunks=[]
rerun_chunks_bins=[]
for chunk in range(len(chunked)):
    count =0
    for i in range(len(chunked[chunk])):
        if chunked[chunk][i]<= 5:
            count +=1
    if count != 0 and count != 5:
        rerun_chunks.append(chunked[chunk])
        rerun_chunks_bins.append(chunk)
        
rerun_chunks, rerun_chunks_bins

In [ ]:
#z=2: 14-17
#z=2.5: 14-18
#z=3:11-16
#z=3.5: 10-14
#z=4:10-14
#z=4.5:10-14
#z=5:11-14
#z=5.5: 10-14
#z=6: 11-14
#z=6.5: 11-14
#z=7: 12:15

## for each redshift, rerun flux bins 10-18 20 times

In [ ]:
#For objects that had a completeness percentage between 0 and 100, run 20 more noise iterations on each (5 is not fine enough)
inds_20=[]
for i in range(11):
    x = (np.arange(10 + (31*i),19+(31*i),1))
    for ind in x:
        inds_20.append(ind)
    
len(inds_20)

In [ ]:
#Run 20x noise iterations with EMCEE
snr_master20 =[]
count=0
for elem in inds_20: #iterate through each z+int_flux element for 99 bins near SNR=5 cutoff

    #move the random seed OUTSIDE the try so it doesn't skip on errors
    np.random.seed(int(time.time()*100)%123456789) #set random seed for 5 noise iterations
    
    try:
        wave_grid = waves[elem]
        fluxes_grid = fluxes[elem]
        noises_grid = flux_errs[elem]

        noisy_fluxes=[]
       # np.random.seed(int(time.time()*100)%123456789) #set random seed for 5 noise iterations

        for i in range(20):
            noise_arr = np.random.normal(0, flux_errs[elem])
            noisy_fluxes.append(fluxes[elem] + noise_arr)

        for noise in range(len(noisy_fluxes)):
            line_window = .3#e4

            #best_lamba_obs = waves[19][ha_centers[19]]
            best_lamba_obs = .6563*(1 + pandeia_simgrid['Redshifts'][elem])

            check_wave, check_flux, check_flux_err, check_df = fitting_line(waves[elem], 
                                                                            noisy_fluxes[noise], flux_errs[elem], 
                                                                            best_lamba_obs, line_window, save_df = False,
                                                                            diagnose = True)
            xarr = np.linspace(check_wave[0], check_wave[-1], 1000)

            l16_params = check_df.quantile(q = 0.16).values[:-2]
            u84_params = check_df.quantile(q = 0.84).values[:-2]

            l16_model = comb(xarr, *l16_params)#, line_center )
            med_model = comb(xarr, *check_df.quantile(q = 0.5).values[:-2])#, line_center )
            u84_model = comb(xarr, *u84_params)#, line_center )

            #Getting the median parameters from the emcee fit
            median_param_vals = check_df.quantile(q = 0.5).values[:-2] #We use the [:-2] because the last two columns 
                                                                       #are LnL and Flux
                                                                       #We do not need those for the line_model 
            print("Median Param Vals: ")
            print(median_param_vals)
            print(median_param_vals[1], median_param_vals[2])

            #max_likely_vals = check_df.iloc[check_df.LnL.argmax(),:-2].values #doing max_likelihood values too

            dot1 = median_param_vals[1] - 3*median_param_vals[2] #Gets the bounds of the wavelength 3-sigma range
            #dot2 = median_param_vals[1]
            dot3 = median_param_vals[1] + 3*median_param_vals[2]




            med_amp = check_df.quantile(q = 0.5)[0]
            low_amp = check_df.quantile(q = 0.16)[0]

            snr_peak = med_amp/(med_amp - low_amp)


            #print("Median: " + str(median_param_vals) + " Max_Likely: " + str(max_likely_vals)) #compare median and max likelihood
            print("Median Integrated Flux: " + str(check_df.quantile(q = 0.5).values[-1]))
            print("Flux from Data: " + str(data_fluxes) + " ± " + str(prop_err))
            print("Baseline: " + str(baseline))

            fig = plt.figure(figsize = (10, 7))
            plt.step(check_wave, check_flux, color = 'cadetblue', label = 'Data', where ='mid')
            plt.scatter(check_wave, check_flux, color = 'cadetblue')
            plt.plot(xarr, comb(xarr, *median_param_vals), label = 'Model', color = "black")

            #plt.plot(xarr, comb(xarr, *max_likely_vals), label = 'Model', color = "red")
            plt.errorbar(check_wave, check_flux, yerr = check_flux_err, fmt = "none", capsize = 3, color = "cadetblue")
            plt.axvline(dot1, color = 'red', linestyle = '--', label = "3-sigma")
            #plt.axvline(dot2, color = 'red', linestyle = '--')
            plt.axvline(dot3, color = 'red', linestyle = '--')
            plt.fill_between(xarr, u84_model, l16_model, color = 'dodgerblue', alpha = 0.5)
            plt.xlabel("Wavelength (Microns)")
            plt.ylabel("Flux (Jy)")
            #plt.ylabel("Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
            #plt.title("First EMCEE Fit for " + str(capstone_rubies['filename'][19]))
            plt.title("Pandeia EMCEE Fit Number 1")
            plt.legend()
            fig.text(.85, .7, 'z = ' + str(pandeia_simgrid[elem]['Redshifts']), ha='center')
            fig.text(.8,.65, 'Integrated Flux = ' + str(pandeia_simgrid[elem]['Integrated Fluxes'])[0:5] + str(pandeia_simgrid[elem]['Integrated Fluxes'])[-4:], ha = 'center')
            fig.text(.8, .6, 'SNR = ' + str(snr_peak)[:5])
            plt.show()
            snr_master20.append(snr_peak)
            count+=1
            print(count)
#             if count%20 ==0:
#                 print(count)
    except Exception as e:
        #err_ind.append(i)
        print("EMCEE Failed for z = " + str(pandeia_simgrid['Redshifts'][elem]) + ', int_flux = ' + str(pandeia_simgrid['Integrated Fluxes'][elem]) + ", Noise Iteration " + str(noisy_fluxes[noise]))
        print(e)
        snr_master20.append(0)
        count+=1
        print(count)
#         if count%20 == 0:
#             print(count)

In [ ]:
#Split these new runs by redshift
chunked20 = [snr_master20[i:i + 20] for i in range(0, len(snr_master20), 20)]

In [ ]:
#z = 2: (5 missing at 1.584e-18) perfect, add the old 5
#z = 2.5: (7 missing at 1e-18) add the old 5 plus 2 more 0 values

#at snr_master20[54], insert these 5 values [2.7296970551656052,
#  3.327422560068393,
#  3.8590296934416632,
#  2.669613186184701,
#  2.874062079647242]

#also, at snr_master20[187], insert two 0s and also insert these 5 values
#[2.5675362931902894,
#  2.857781216592599,
#  0,
#  2.2832190218635695,
#  3.801251190104489]

In [ ]:
#replace SNR = nan with SNR = 0
nan_inds20=[]
for i in range(len(snr_master20)):
    if math.isnan(snr_master20[i]):
        nan_inds20.append(i)
        
for i in nan_inds20:
    snr_master20[i] = 0

In [ ]:
#Calculate which bins still had a completeness value between 0 and 100

list_of = []
for chunk in chunked20:
    num_obs = 0
    for i in chunk:
        if i >=5.0:
            num_obs +=1
    list_of.append(num_obs)

zbin_listof = [list_of[i:i+9] for i in range(0, len(list_of), 9)]
zbin_listof #shows the number of sources in the central bins 

In [ ]:
#Create a color wheel for redshift
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', 'black']

#Complete the rest of the bins with either 0% or 100% completeness
#DONT RUN AGAIN


for i in zbin_listof:
    count0 = 0
    while count0 <10:
        i.insert(0,0)
        count0+=1
    count20=0
    while count20 <12:
        i.insert(-1,20)
        count20+=1


In [ ]:
#Sample plot, not important
for i in range(len(zbin_listof)):
    plt.figure()
    plt.plot(pandeia_simgrid['Integrated Fluxes'][0:31], np.array(zbin_listof[i])/20, color = colors[i])
   # plt.hist(np.log10(zbin_listof[i]), bins = 11)
    plt.xlabel("Integrated Flux")
    plt.xscale('log')
    plt.ylabel("Completeness Percentage")
    plt.title("Completeness Percentage of Sample at z = " + str(zs[i]))
    plt.show()
    

In [ ]:
#Plot the completeness percentage of our simulation grid (the percentage of objects that EMCEE succeessfully fit a Gaussian
#function to with a SNR>5 as a function of luminosity bins for each redshift bin
plt.figure()
for i in range(len(zbin_listof)):
    plt.plot(pandeia_simgrid['Integrated Fluxes'][0:31], np.array(zbin_listof[i])/20, alpha = .66, color = colors[i], label = "z = " + str(zs[i]))
   # plt.hist(np.log10(zbin_listof[i]), bins = 11)
    plt.xlabel("Integrated Flux")
    plt.xscale('log')
    plt.ylabel("Completeness Percentage")
  #  plt.xlim(10e-18, 10e-16)
    plt.title("Completeness Percentage of Sample at z = " + str(zs[i]))
    plt.legend()
plt.show()

## for each redshift, rerun flux bins 10-16 80 times, bringing them to a total of 100 times


In [ ]:
#Rerun those non 0/100% completeness bins with 100 noise iterations
inds_80=[]
for i in range(11):
    x = (np.arange(10 + (31*i),17+(31*i),1))
    for ind in x:
        inds_80.append(ind)
    
len(inds_80)

In [ ]:
#Running EMCEE with x100 noise
snr_master80 =[]
count=0
np.random.seed(int(time.time()*100)%123456789) #set random seed for 5 noise iterations
for elem in inds_80: #iterate through each z+int_flux element for 99 bins near SNR=5 cutoff

    #move the random seed OUTSIDE the try so it doesn't skip on errors
    
    
    wave_grid = waves[elem]
    fluxes_grid = fluxes[elem]
    noises_grid = flux_errs[elem]

    noisy_fluxes=[]
       # np.random.seed(int(time.time()*100)%123456789) #set random seed for 5 noise iterations

    for i in range(80):
        noise_arr = np.random.normal(0, flux_errs[elem])
        noisy_fluxes.append(fluxes[elem] + noise_arr)

    for noise in range(len(noisy_fluxes)):
        try:
            line_window = .3#e4

            #best_lamba_obs = waves[19][ha_centers[19]]
            best_lamba_obs = .6563*(1 + pandeia_simgrid['Redshifts'][elem])

            check_wave, check_flux, check_flux_err, check_df = fitting_line(waves[elem], 
                                                                            noisy_fluxes[noise], flux_errs[elem], 
                                                                            best_lamba_obs, line_window, save_df = False,
                                                                            diagnose = True)
            xarr = np.linspace(check_wave[0], check_wave[-1], 1000)

            l16_params = check_df.quantile(q = 0.16).values[:-2]
            u84_params = check_df.quantile(q = 0.84).values[:-2]

            l16_model = comb(xarr, *l16_params)#, line_center )
            med_model = comb(xarr, *check_df.quantile(q = 0.5).values[:-2])#, line_center )
            u84_model = comb(xarr, *u84_params)#, line_center )

            #Getting the median parameters from the emcee fit
            median_param_vals = check_df.quantile(q = 0.5).values[:-2] #We use the [:-2] because the last two columns 
                                                                       #are LnL and Flux
                                                                       #We do not need those for the line_model 
            print("Median Param Vals: ")
            print(median_param_vals)
            print(median_param_vals[1], median_param_vals[2])

            #max_likely_vals = check_df.iloc[check_df.LnL.argmax(),:-2].values #doing max_likelihood values too

            dot1 = median_param_vals[1] - 3*median_param_vals[2] #Gets the bounds of the wavelength 3-sigma range
            #dot2 = median_param_vals[1]
            dot3 = median_param_vals[1] + 3*median_param_vals[2]




            med_amp = check_df.quantile(q = 0.5)[0]
            low_amp = check_df.quantile(q = 0.16)[0]

            snr_peak = med_amp/(med_amp - low_amp)


            #print("Median: " + str(median_param_vals) + " Max_Likely: " + str(max_likely_vals)) #compare median and max likelihood
            print("Median Integrated Flux: " + str(check_df.quantile(q = 0.5).values[-1]))
            print("Flux from Data: " + str(data_fluxes) + " ± " + str(prop_err))
            print("Baseline: " + str(baseline))

            fig = plt.figure(figsize = (10, 7))
            plt.step(check_wave, check_flux, color = 'cadetblue', label = 'Data', where ='mid')
            plt.scatter(check_wave, check_flux, color = 'cadetblue')
            plt.plot(xarr, comb(xarr, *median_param_vals), label = 'Model', color = "black")

            #plt.plot(xarr, comb(xarr, *max_likely_vals), label = 'Model', color = "red")
            plt.errorbar(check_wave, check_flux, yerr = check_flux_err, fmt = "none", capsize = 3, color = "cadetblue")
            plt.axvline(dot1, color = 'red', linestyle = '--', label = "3-sigma")
            #plt.axvline(dot2, color = 'red', linestyle = '--')
            plt.axvline(dot3, color = 'red', linestyle = '--')
            plt.fill_between(xarr, u84_model, l16_model, color = 'dodgerblue', alpha = 0.5)
            plt.xlabel("Wavelength (Microns)")
            plt.ylabel("Flux (Jy)")
            #plt.ylabel("Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
            #plt.title("First EMCEE Fit for " + str(capstone_rubies['filename'][19]))
            plt.title("Pandeia EMCEE Fit Number 1")
            plt.legend()
            fig.text(.85, .7, 'z = ' + str(pandeia_simgrid[elem]['Redshifts']), ha='center')
            fig.text(.8,.65, 'Integrated Flux = ' + str(pandeia_simgrid[elem]['Integrated Fluxes'])[0:5] + str(pandeia_simgrid[elem]['Integrated Fluxes'])[-4:], ha = 'center')
            fig.text(.8, .6, 'SNR = ' + str(snr_peak)[:5])
            plt.show()
            snr_master80.append(snr_peak)
            count+=1
            print(count)
#             if count%20 ==0:
#                 print(count)
        except Exception as e:
            #err_ind.append(i)
            print("EMCEE Failed for z = " + str(pandeia_simgrid['Redshifts'][elem]) + ', int_flux = ' + str(pandeia_simgrid['Integrated Fluxes'][elem]) + ", Noise Iteration " + str(noisy_fluxes[noise]))
            print(e)
            snr_master80.append(0)
            count+=1
            print(count)
    #         if count%20 == 0:
#             print(count)

In [ ]:
#total SNR calculations for the 100x noise iterations
snr_master80 = list(Table.read('snar_master80.fits')['snr_master80'])

In [ ]:
#Split these by redshift
chunked80 = [snr_master80[i:i + 80] for i in range(0, len(snr_master80), 80)]

nan_inds80=[]
for i in range(len(snr_master80)):
    if math.isnan(snr_master80[i]):
        nan_inds80.append(i)
        
for i in nan_inds80:
    snr_master80[i] = 0


In [ ]:
#Find the completeness percentage for the remaining central 7 luminosity bins for each redshift that we ran noise x100

list_of80 = []
for chunk in chunked80:
    num_obs = 0
    for i in chunk:
        if i >=5.0:
            num_obs +=1
    list_of80.append(num_obs)

zbin_listof80 = ([list_of80[i:i+7] for i in range(0, len(list_of80), 7)])
zbin_listof80 #shows the number of sources in the central bins 


In [ ]:
#From the 20 noise iterations for 9 bins

zbin_listof = [[0, 0, 0, 0, 1, 5, 12, 20, 20],
 [1, 0, 1, 2, 9, 17, 20, 20, 20],
 [0, 0, 2, 7, 18, 20, 20, 20, 20],
 [0, 5, 9, 19, 20, 20, 20, 20, 20],
 [3, 7, 18, 20, 20, 20, 20, 20, 20],
 [4, 10, 20, 20, 20, 20, 20, 20, 20],
 [4, 17, 20, 20, 20, 20, 20, 20, 20],
 [4, 15, 20, 20, 20, 20, 20, 20, 20],
 [1, 7, 16, 19, 20, 20, 20, 20, 20],
 [2, 10, 18, 20, 20, 20, 20, 20, 20],
 [0, 3, 10, 17, 20, 20, 20, 20, 20]]

In [ ]:
#Combining the x5, x20, and x100 noise iterations
need=[]
for i in zbin_listof:
    i = i[:-2]
    need.append(i)
need

In [ ]:
#Calculate the final completeness percentages for ALL the smiulation bins, including 0, 100, and those in between

for i in need: 
    count0 = 0 
    while count0 <10:
        i.insert(0,0)
        count0+=1
    count20=0
    while count20 <14:
        i.insert(-1,20)
        count20+=1

for i in zbin_listof80:
    count0 = 0
    while count0 <10:
        i.insert(0,0)
        count0+=1
    count20=0
    while count20 <14:
        i.insert(-1,80)
        count20+=1

final_comp = np.array(zbin_listof80) + np.array(need)

final_comp[0][16], final_comp[0][-1] = final_comp[0][-1], final_comp[0][16]
final_comp

In [ ]:
#Save the completeness percentages in this array! These can now be applied to my dataset
final_comp = array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   5,  24,  66, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1,   0,   5,
         14,  52,  89, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   3,  16,
         49,  92, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   5,  14,  50,
         96, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  13,  47,  89,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  24,  59,  99,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  31,  81,  96,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  22,  67,  98,
        100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  16,  40,  88,
         98, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  10,  44,  80,
         98, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   2,  14,  42,
         96,  99,  98, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
        100, 100, 100, 100, 100]])

In [ ]:
#Combine the last two redshift bins
final_comp = final_comp[:-1]

final_comp_new=[]
for i in range(5):
    final_comp_new.append((final_comp[0+(2*i)]+final_comp[1+(2*i)])/2)
final_comp_new

In [ ]:
#Plot the different LD-completeness percentages for different redshifts
for i in range(len(final_comp_new)):
    plt.figure()
    plt.plot(pandeia_simgrid['Integrated Fluxes'][0:31], np.array(final_comp_new[i])/100, color = colors[i])
    plt.scatter(pandeia_simgrid['Integrated Fluxes'][0:31], np.array(final_comp_new[i])/100, color = colors[i])
   # plt.hist(np.log10(zbin_listof[i]), bins = 11)
    plt.xlabel("Integrated Flux")
    plt.xscale('log')
    plt.ylabel("Completeness Percentage")
    plt.title("Completeness Percentage of Sample at z = " + str(zs[i]))
    plt.show()



In [ ]:
#Plot with evolving redshift
plt.figure()
for i in range(len(final_comp_new)):
    plt.plot(pandeia_simgrid['Integrated Fluxes'][0:31], np.array(final_comp_new[i])/100, alpha = .66, color = colors[i], label = "z = " + str(zs[i]))
   # plt.hist(np.log10(zbin_listof[i]), bins = 11)
    plt.xlabel("Integrated Flux")
    plt.xscale('log')
    plt.ylabel("Completeness Percentage")
  #  plt.xlim(10e-18, 10e-16)
    plt.title("Completeness Percentage of Sample")
    plt.legend()
plt.show()


# Now, line detection completeness correction is done, we have to convert it into luminosity space before we can apply it to the sample!

In [ ]:
#First, convert the 31 integrated flux bins into luminosity bins for 11 redshift bins. 341 UNIQUE lum bins!
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0 = 70, Om0 =  0.3, Tcmb0 = 2.725)


In [ ]:

#convert flux to luminosity for new histogram
lum_bins=[]
for i in range(len(pandeia_simgrid)):
    z_ld = pandeia_simgrid['Redshifts'][i]
    ld = (cosmo.luminosity_distance(z_ld)).to(u.cm) #Uses astropy.cosmology to calculate luminosity distance in cm
    lum = 4*np.pi*(ld**2)*pandeia_simgrid['Integrated Fluxes'][i] #L = 4pi*d^2*f, in erg/s/cm^2... this should be in erg/s 
    lum_bins.append(lum.value)

len(lum_bins)


In [ ]:
#Plot the line detection completeness as a function of luminosity now!
count=1
for i in range(len(final_comp_new)):
    plt.figure()
    plt.plot(np.log10(lum_bins[0 + (31*count):31+(31*count)]), np.array(final_comp_new[i])/100, color = colors[i])
    plt.scatter(np.log10(lum_bins[0+(31*count):31+(31*count)]), np.array(final_comp_new[i])/100, color = colors[i])
   # plt.hist(np.log10(zbin_listof[i]), bins = 11)
    plt.xlabel("Luminosity (erg/s)")
    plt.xscale('log')
    plt.ylabel("Completeness Percentage")
    plt.title("Completeness Percentage of Sample at z = " + str(zs[i]))
    plt.show()
    count+=2 #This means it is going my every OTHER redshift bin, so centered at 2.5, 3.5, 4.5, 5.5, 6.5.


In [ ]:
#Now plot with evolving redshift
plt.figure()
count=1
final_lumbins=[]
for i in range(len(final_comp_new)):
    final_lumbins.append(np.log10(lum_bins[0+(31*count):31+(31*count)]))
    plt.plot(np.log10(lum_bins[0+(31*count):31+(31*count)]), np.array(final_comp_new[i])/100, alpha = .66, color = colors[i], label = "z = " + str(zs[i]))
   # plt.hist(np.log10(zbin_listof[i]), bins = 11)
    plt.xlabel("Luminosity (erg/s)")
    plt.xscale('log')
    plt.ylabel("Completeness Percentage")
  #  plt.xlim(10e-18, 10e-16)
    plt.title("Completeness Percentage of Sample")
    plt.legend()
    count+=2
plt.show()
final_lumbins = np.array(final_lumbins)

In [ ]:
#Interpolate the completeness functions
plt.figure(figsize = (18,10), dpi = 250)
binedges = np.arange(40,45.25,.1) #i can change this up here to change the bin precision
bincenters=binedges[1:]-(binedges[1]-binedges[0])/2
comp_vals=[]
for i in range(5):
    #Cool, fill_value allows me to extend the min and max infinitely outward and not have bound errors
    interp_func = interp1d(final_lumbins[i], final_comp_new[i], kind ='linear', fill_value = (final_comp_new[i][0], final_comp_new[i][-1]), bounds_error = False)
    x_new = np.linspace(min(final_lumbins[i]), max(final_lumbins[i]), 1000) #1000 for even interpolation
    y_new = interp_func(x_new) 
    plt.plot(x_new, y_new, color = colors[i], label = f'z={zs[i]-0.5}-{zs[i]+0.5}', linewidth = 3)
    plt.xlabel("Luminosity (erg/s)", fontsize = 35)
    plt.ylabel("Completeness Percentage", fontsize = 35)
    plt.title("Line Detection Completeness with Pandeia",fontsize = 40)
    plt.tick_params(axis = "both", labelsize = 20)
    plt.legend(fontsize = 30)
    comp_vals.append(interp_func(bincenters)) #Creates consistent completion lum bins for all
comp_vals = np.array(comp_vals)

In [ ]:
#Return the completion values
comp_vals

In [ ]:
#Save the completion values for different bin widths

comp_vals = array([[  0.        ,   0.        ,   0.14074998,   1.95374991,
         36.38199896,  96.28549937, 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          7.08094019,  78.69201576, 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,  41.70939811, 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  76.9187532 , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   5.70662349,  81.43678359,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ]])


#.1 lum bins version!
# comp_vals = array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 2.65749981e-01, 2.34250019e-01,
#         1.32874991e+00, 4.89174983e+00, 1.84272492e+01, 4.33819990e+01,
#         7.05847490e+01, 9.20354994e+01, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02],
#        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 3.37250808e-02, 2.58094019e+00, 8.83050579e+00,
#         3.35328563e+01, 7.28170158e+01, 9.60539601e+01, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02],
#        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         7.82061928e+00, 3.30843981e+01, 7.03321833e+01, 9.65364171e+01,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02],
#        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 9.98791129e+00, 4.44028599e+01,
#         8.26687532e+01, 9.81307069e+01, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02],
#        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
#         8.95662349e+00, 3.29801601e+01, 7.09367836e+01, 9.36455945e+01,
#         9.93779421e+01, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02,
#         1.00000000e+02, 1.00000000e+02, 1.00000000e+02, 1.00000000e+02]])
comp_vals

# Now, we will be applying it to the sample! First, plot the lower limit LFs again, and then plot the LD-corrected LFs (still technically a lower lim)!

In [ ]:
#Plot lower limit LFs from the previous notebook
eiftsnr5 = Table.read("eiftsnr5wlum.html")
eiftsnr5

In [ ]:
#read in the weighted data from obs_correction notebook

rubies_weights = Table.read("rubies_weights3.html")
rubies_weights.rename_column("filename", "File Name")  # Standardize column names
rubies_weights

In [ ]:
#combine the weights from the obs_correction into this notebook, now we can weight each object accordingly
from astropy.table import join
weighted_eift = join(rubies_weights, eiftsnr5, keys="File Name", join_type="right")
display(weighted_eift)

# Count the number of sources with a valid "Weight"
valid_weight_mask = ~np.isnan(weighted_eift["Weight"])
num_sources_with_weight = np.sum(valid_weight_mask)
print(num_sources_with_weight) #448/504 objects have weights


#Change the nan weights to 1.0
weighted_eift["Weight"].fill_value = 1 
weighted_eift["Weight"] = weighted_eift["Weight"].filled()

# Count the number of sources with a valid "Weight"
valid_weight_mask = ~np.isnan(weighted_eift["Weight"])
num_sources_with_weight = np.sum(valid_weight_mask)
print(num_sources_with_weight) #448/504 objects have weights

eiftsnr5 = weighted_eift
eiftsnr5


In [ ]:
#Reestablish redshift bins
zs = np.arange(2.5,7.5)
zs

In [ ]:
#Apply the observational weights to each object in my sample
counts=[]
color_ind=0
binedges=np.arange(40,45.25,0.25)
bincenters=binedges[1:]-(binedges[1]-binedges[0])/2
for z in zs:
    bins=np.arange(40,45.25,0.1)
    histdata,binedges,_=plt.hist(np.log10(eiftsnr5['Luminosity'][np.abs(eiftsnr5['Redshift']-z)<0.5]),bins=binedges, alpha = .5, label = str(z-.5) + " < z < " + str(z+.5), color = colors[color_ind])
    counts.append(histdata)
    plt.title("Lower Limit LF-Hist at z = " + str(z))
    plt.xlabel("Log(Luminosity) (erg s$^{-1}$)")
    plt.ylabel("Number")
    plt.legend()
    plt.show()
    color_ind+=1
counts=np.array(counts)

In [ ]:
np.array(final_comp_new)

In [ ]:
#Combine redshift bins at the high-z end
counts[-2] = counts[-2]+counts[-1]
counts = counts[:-1]
counts

In [ ]:
#Plot scatter number density for the lower limit LF-histogram (pre-corrections)
for num, i in enumerate(zs[:-1]):
    plt.scatter(bincenters,counts[num],label=str(i))
    plt.errorbar(bincenters, counts[num], yerr = og_errs[num], fmt= 'o')
    plt.xlabel("Luminosity (erg/s)")
    plt.ylabel("Number")
    plt.title("Scatter version of Lower-Limit LF-Hist")
plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
#Copied over from previous notebook to get volume density

#the angular distance covered by NIRSpec in RUBIES in both EGS and UDS fields
egs_angle = 82.7339 #8888 repeating...
uds_angle = 73.1375
tot_angle = egs_angle+uds_angle #in arcmin

tot_angle /=3600 #total angle in degrees

#What fraction of the sky is the RUBIES survey looking at
rubies_frac = tot_angle/41252.96125 #back to arcminutes
rubies_frac
zs=np.arange(2.5,7.5,1)

In [ ]:
#Apply bin width and volume density division

for num,z in enumerate(zs):
    #divide by binwidth and volume of redshift bin
    counts[num]=counts[num]/(bincenters[1]-bincenters[0])/((cosmo.comoving_volume(z+0.5)-cosmo.comoving_volume(z-0.5))*rubies_frac)
    og_errs[num]=og_errs[num]/(bincenters[1]-bincenters[0])/((cosmo.comoving_volume(z+0.5)-cosmo.comoving_volume(z-0.5))*rubies_frac)
#    

In [ ]:
#Plot the proper lower limit LF (pre-correction)

#THIS PLOT WILL BE FOR PRE-CORRECTIONS

colorind=0
for num, i in enumerate(zs[:-1]):
   # plt.scatter(bincenters,np.log10(counts[num]),label=str(i), color = colors[colorind])
    plt.errorbar(bincenters, counts[num], yerr = og_errs[num], label = str(i), color = colors[colorind], fmt = 'o')
    plt.xlabel("Log(Lumninosity) (erg/s)")
    plt.ylabel("Log(ϕ) (Mpc$^{-3}$dlogL$^{-1}$)")
    plt.title("Lower Limit LF with Evolving Redshift")
    colorind+=1
plt.yscale('log')
plt.legend()
plt.show()

# Now, time to apply the obs_corrections!

In [ ]:
#Actually multiplying out the weights to each individual object that is in my sample
weight_counts=[]
color_ind=0
binedges=np.arange(40,45.25,0.25)
bincenters=binedges[1:]-(binedges[1]-binedges[0])/2
for z in zs:
    bins=np.arange(40,45.25,0.1)
    histdata,binedges,_=plt.hist(np.log10(eiftsnr5['Luminosity'][np.abs(eiftsnr5['Redshift']-z)<0.5]),weights = eiftsnr5['Weight'][np.abs(eiftsnr5['Redshift']-z)<.5],bins=binedges, alpha = .5, label = str(z-.5) + " < z < " + str(z+.5), color = colors[color_ind])
    weight_counts.append(histdata)
    plt.title("OBS-corrected LF-Hist at z = " + str(z))
    plt.xlabel("Log(Luminosity) (erg s$^{-1}$)")
    plt.ylabel("Number")
    plt.legend()
    plt.show()
    color_ind+=1
weight_counts=np.array(weight_counts)

In [ ]:
#Plot the obs-corrected scatter plot version of the LF-histogram
for num, i in enumerate(zs[:-1]):
    plt.scatter(bincenters,weight_counts[num],label=str(i))
    plt.errorbar(bincenters, weight_counts[num], yerr = obs_errs[num], fmt = 'o')
    plt.xlabel("Luminosity (erg/s)")
    plt.ylabel("Number")
    plt.title("Scatter version of Obs-Corrcted LF-Hist")
plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
#Divide by bin width and volume
for num,z in enumerate(zs):
    #divide by binwidth and volume of redshift bin
    weight_counts[num]=weight_counts[num]/(bincenters[1]-bincenters[0])/((cosmo.comoving_volume(z+0.5)-cosmo.comoving_volume(z-0.5))*rubies_frac)

#Plot the obs-corrected proper LF
colorind=0
for num, i in enumerate(zs[:-1]):
    plt.scatter(bincenters,np.log10(weight_counts[num]),label=str(i), color = colors[colorind])
    plt.xlabel("Log(Lumninosity) (erg/s)")
    plt.ylabel("Log(ϕ) (Mpc$^{-3}$dlogL$^{-1}$)")
    plt.title("Obs-Corrcted LF with Evolving Redshift")
    colorind+=1
#plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
#Combining the redshifts>7 into the 6-7 range, since I did run simulations at redshifts above 7


weight_counts[-2] = weight_counts[-2]+weight_counts[-1]
weight_counts = weight_counts[:-1]

In [ ]:


#CHANGING COMPVALS TO IGNORE ANYTHING UNDER 10% COMPLETION, as we cannot trust it, save these new values
comp_vals = array([[  0.        ,   0.        ,   0.,   0.,
         36.38199896,  96.28549937, 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.,  78.69201576, 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,  41.70939811, 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  76.9187532 , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.,  81.43678359,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ,
        100.        , 100.        , 100.        , 100.        ]])
counts_cor

counts_cor = weight_counts/(comp_vals/100)

In [ ]:
#Apply line-detection completeness to sample

binedges=np.arange(40,45.25,0.25)
bincenters=binedges[1:]-(binedges[1]-binedges[0])/2

for z in range(5):
    plt.figure()
    plt.step(bincenters, counts_cor[z], where = "mid", color = colors[z], label = str((z+2.5)-.5) + " < z < " + str((z+2.5)+.5))
    plt.fill_between(bincenters, counts_cor[z], step='mid', alpha=.5, color=colors[z], label="Fill Area")
    plt.xlabel("Luminosity (erg/s)")
    plt.ylabel("Number")
    plt.title("LD-corrected LF-Hist at z = " + str(z+2.5))
    plt.show()
    #plt.legend()


In [ ]:
#Plot LD-corrected scatter version of LF-histogram

for num, i in enumerate(zs[:-1]):
    plt.scatter(bincenters,counts_cor[num],label=str(i))
    plt.errorbar(bincenters, counts_cor[num], yerr = errs[num], fmt = 'o')
    plt.xlabel("Log Luminosity (erg/s)")
    plt.ylabel("Number")
    plt.title("Scatter version of LD-Corrected LF-Hist")
plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
zs = np.arange(2.5, 7.5)
zs

In [ ]:
#Now, again divide by binwidth and volume to get the final LD-corrected LF!
for num,z in enumerate(zs):
    #divide by binwidth and volume of redshift bin
    counts_cor[num]=counts_cor[num]/(bincenters[1]-bincenters[0])/((cosmo.comoving_volume(z+0.5)-cosmo.comoving_volume(z-0.5))*rubies_frac)
    errs[num]=errs[num]/(bincenters[1]-bincenters[0])/((cosmo.comoving_volume(z+0.5)-cosmo.comoving_volume(z-0.5))*rubies_frac)
    

In [ ]:
#Plot the final proper obs and LD-corrected LFs!! Fits to Schechter functions are included even though they are calculated later
#Subplot separation indicates different redshift bins

#0.25 bin version!
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10.5), dpi=150)  # Create a 2x3 grid
axes = axes.flatten() 
xarr = np.linspace(39.9, 46.25, 1000)

colorind=0
for num, i in enumerate(zs):#in enum, first is elem, second id index
    ax = axes[num]
    yarr=[]
    for x in xarr:
        yarr.append(sobral_schechter_log_inputs(x, schechter_fits[num][0], schechter_fits[num][1], schechter_fits[num][2])) #schechter fits

    ax.errorbar(bincenters, counts[num], yerr = og_errs[num], fmt = 'o', label = "Raw Data", color =  colors[colorind], alpha = 0.3)
    ax.errorbar(bincenters+(num-3)/100, (counts_cor[num]), yerr = errs[num], fmt = 'o',label= "Corrected Data", color = colors[colorind])
    ax.plot(xarr, yarr, color = colors[num])
    ax.set_xlabel("Log(L) (erg/s)", fontsize = 12)
    ax.set_ylabel("Log(ϕ) (Mpc$^{-3}$dlogL$^{-1}$)", fontsize = 12)
    ax.set_title("H-α Line Luminosity Function (" + f'z={i-0.5}-{i+0.5}' + ")", fontsize = 14)
    ax.set_yscale('log')
    ax.tick_params(axis='both')#, labelsize=17)  # Changes font size for both x and y ticks
    ax.legend(fontsize = 11)
    ax.set_xlim(39.9, 44.25)
    ax.set_ylim(1e-6, .02)
    colorind+=1
fig.delaxes(axes[-1]) 
plt.show()


# for z in range(len(xs)):
#   #  xarr = np.linspace(min(xs[z]), max(xs[z]), 1000)
#     plt.plot(xarr, yarr, label = f'z={(z+2.5)-0.5}-{(z+2.5)+0.5}', color = colors[z])
#     #plt.ylim(1e-6, .02)

In [ ]:
#Rubies 42046 is the z=5-6 bright, AGN candidate... for another time


In [ ]:
#Save all the counts, through each correction, below
counts = array([[ 0.,  0.,  0.,  4., 10., 11., 15., 16., 10.,  6.,  0.,  1.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  1., 16., 41., 40., 31., 28., 14.,  7.,  4.,  1.,
         2.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  7., 17., 17., 22., 14., 13.,  5.,  2.,  1.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  3., 10., 19., 22., 16., 13.,  6.,  4.,  0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  0.,  5., 12.,  7., 15.,  7.,  4.,  1.,  1.,
         0.,  1.,  0.,  1.,  0.,  0.,  0.]])


weight_counts = array([[  0.        ,   0.        ,   0.        , 339.65640709,
        551.47084808, 370.95093525, 668.67386895, 681.78923091,
        309.53818049, 216.37851662,   0.        ,   1.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,  20.96078431,
        287.60674386, 867.51030517, 624.90918344, 804.44184856,
        332.90703374, 132.88617191,  73.09205426,  26.875     ,
         10.91666667,   7.25      ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
        143.69685858, 408.62537057, 241.26602907, 518.23794519,
        162.15498874, 102.18636364,  53.63176471,  19.08727273,
          1.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
         31.22177419, 239.45429499, 251.65956221, 379.60763027,
        174.95698925, 100.50595238,  71.94688645,  24.35714286,
          0.        ,   0.        ,   0.        ,   2.66666667,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,  26.        , 176.40232959,  58.72905526,
        225.71962014,  89.97593583,  25.        ,  13.5       ,
         13.5       ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ,   0.        ]])

#note, this is pre-vol div, run vol div after for proper data, after 10% cut
counts_cor = array([[           nan,            nan,            nan,            inf,
        1.51577941e+03, 3.85261475e+02, 6.68673869e+02, 6.81789231e+02,
        3.09538180e+02, 2.16378517e+02, 0.00000000e+00, 1.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            inf,
                   inf, 1.10241210e+03, 6.24909183e+02, 8.04441849e+02,
        3.32907034e+02, 1.32886172e+02, 7.30920543e+01, 2.68750000e+01,
        1.09166667e+01, 7.25000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            nan,
                   inf, 9.79696157e+02, 2.41266029e+02, 5.18237945e+02,
        1.62154989e+02, 1.02186364e+02, 5.36317647e+01, 1.90872727e+01,
        1.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            nan,
                   inf,            inf, 3.27175821e+02, 3.79607630e+02,
        1.74956989e+02, 1.00505952e+02, 7.19468864e+01, 2.43571429e+01,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 2.66666667e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            nan,
                   nan,            inf,            inf, 7.21161282e+01,
        2.25719620e+02, 8.99759358e+01, 2.50000000e+01, 1.35000000e+01,
        1.35000000e+01, 0.00000000e+00, 1.00000000e+00, 0.00000000e+00,
        1.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])

# Before fitting, we need to calculate Uncertainties!

In [ ]:
#FROM EARLIER, reestablish bins
binedges = np.arange(40,45.25,.25) #i can change this up here to change the bin precision
bincenters=binedges[1:]-(binedges[1]-binedges[0])/2
bincenters

In [ ]:
#Use square root errors, and run them through each mathematical process (volume conversions done with enumerate cells)
og_errs = sqrt(counts) #We will just be using a simple square root error 

obs_factor = weight_counts/counts #this shows the factor for each lum_bin that counts was multiplied by for obs
obs_errs = obs_factor*og_errs #error values after obs_correction

ld_factor = counts_cor/weight_counts #this shows the factor for each lum_bin the counts were multiplied by for LD
errs = obs_errs*ld_factor #error values after LD_correction
   
og_errs, obs_errs, errs

# Saving the error arrays below

In [ ]:
og_errs = array([[0.        , 0.        , 0.        , 2.        , 3.16227766,
         3.31662479, 3.87298335, 4.        , 3.16227766, 2.44948974,
         0.        , 1.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 1.        , 4.        ,
         6.40312424, 6.32455532, 5.56776436, 5.29150262, 3.74165739,
         2.64575131, 2.        , 1.        , 1.41421356, 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 2.64575131,
         4.12310563, 4.12310563, 4.69041576, 3.74165739, 3.60555128,
         2.23606798, 1.41421356, 1.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 1.73205081,
         3.16227766, 4.35889894, 4.69041576, 4.        , 3.60555128,
         2.44948974, 2.        , 0.        , 0.        , 0.        ,
         1.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         2.23606798, 3.46410162, 2.64575131, 3.87298335, 2.64575131,
         2.        , 1.        , 1.        , 0.        , 1.        ,
         0.        , 1.        , 0.        , 0.        , 0.        ]])

obs_errs = array([[         nan,          nan,          nan, 169.82820355,
         174.39039431, 111.84591526, 172.65085057, 170.44730773,
          97.88456731,  88.3361595 ,          nan,   1.        ,
                  nan,          nan,          nan,          nan,
                  nan,          nan,          nan,          nan],
        [         nan,          nan,          nan,  20.96078431,
          71.90168596, 135.48234784,  98.80681752, 144.48202117,
          62.91351578,  35.51532334,  27.62619977,  13.4375    ,
          10.91666667,   5.12652416,          nan,          nan,
                  nan,          nan,          nan,          nan],
        [         nan,          nan,          nan,          nan,
          54.31230743,  99.10620966,  58.51560716, 110.48870116,
          43.33774367,  28.34139798,  23.98485433,  13.49673998,
           1.        ,          nan,          nan,          nan,
                  nan,          nan,          nan,          nan],
        [         nan,          nan,          nan,          nan,
          18.02589973,  75.72209677,  57.73466315,  80.93261871,
          43.73924731,  27.87533575,  29.3721934 ,  12.17857143,
                  nan,          nan,          nan,   2.66666667,
                  nan,          nan,          nan,          nan],
        [         nan,          nan,          nan,          nan,
                  nan,  11.62755348,  50.92296624,  22.19749642,
          58.28055531,  34.00770717,  12.5       ,  13.5       ,
          13.5       ,          nan,   1.        ,          nan,
           1.        ,          nan,          nan,          nan]])

errs = array([[           nan,            nan,            nan, 8.69242285e+03,
         4.79331537e+02, 1.16160705e+02, 1.72650851e+02, 1.70447308e+02,
         9.78845672e+01, 8.83361597e+01,            nan, 1.00000000e+00,
                    nan,            nan,            nan,            nan,
                    nan,            nan,            nan,            nan],
        [           nan,            nan,            nan,            inf,
         1.01542569e+03, 1.72167845e+02, 9.88068175e+01, 1.44482021e+02,
         6.29135158e+01, 3.55153234e+01, 2.76261998e+01, 1.34375000e+01,
         1.09166667e+01, 5.12652416e+00,            nan,            nan,
                    nan,            nan,            nan,            nan],
        [           nan,            nan,            nan,            nan,
                    inf, 2.37611220e+02, 5.85156071e+01, 1.10488701e+02,
         4.33377437e+01, 2.83413981e+01, 2.39848543e+01, 1.34967400e+01,
         1.00000000e+00,            nan,            nan,            nan,
                    nan,            nan,            nan,            nan],
        [           nan,            nan,            nan,            nan,
                    inf,            inf, 7.50592811e+01, 8.09326187e+01,
         4.37392472e+01, 2.78753356e+01, 2.93721934e+01, 1.21785714e+01,
                    nan,            nan,            nan, 2.66666667e+00,
                    nan,            nan,            nan,            nan],
        [           nan,            nan,            nan,            nan,
                    nan,            inf, 8.92348451e+02, 2.72573344e+01,
         5.82805553e+01, 3.40077072e+01, 1.25000000e+01, 1.35000000e+01,
         1.35000000e+01,            nan, 1.00000000e+00,            nan,
         1.00000000e+00,            nan,            nan,            nan]])

#So, this ratio holds throught for all 3: og_errs/counts, obs_errs/weight_counts, errs/counts_cor (Pre-vol div)

errs_ratio = array([[       nan,        nan,        nan, 0.5       , 0.31622777,
        0.30151134, 0.25819889, 0.25      , 0.31622777, 0.40824829,
               nan, 1.        ,        nan,        nan,        nan,
               nan,        nan,        nan,        nan,        nan],
       [       nan,        nan,        nan,        nan, 0.25      ,
        0.15617376, 0.15811388, 0.1796053 , 0.18898224, 0.26726124,
        0.37796447, 0.5       , 1.        , 0.70710678,        nan,
               nan,        nan,        nan,        nan,        nan],
       [       nan,        nan,        nan,        nan,        nan,
        0.24253563, 0.24253563, 0.21320072, 0.26726124, 0.2773501 ,
        0.4472136 , 0.70710678, 1.        ,        nan,        nan,
               nan,        nan,        nan,        nan,        nan],
       [       nan,        nan,        nan,        nan,        nan,
               nan, 0.22941573, 0.21320072, 0.25      , 0.2773501 ,
        0.40824829, 0.5       ,        nan,        nan,        nan,
        1.        ,        nan,        nan,        nan,        nan],
       [       nan,        nan,        nan,        nan,        nan,
               nan, 0.28867513, 0.37796447, 0.25819889, 0.37796447,
        0.5       , 1.        , 1.        ,        nan, 1.        ,
               nan, 1.        ,        nan,        nan,        nan]])
             
errs_final = array([[           nan,            nan,            nan, 6.72190935e-02,
        3.70670318e-03, 8.98278584e-04, 1.33512070e-03, 1.31808055e-03,
        7.56947975e-04, 6.83109494e-04,            nan, 7.73306759e-06,
                   nan,            nan,            nan,            nan,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            inf,
        8.26370140e-03, 1.40113026e-03, 8.04106144e-04, 1.17581847e-03,
        5.12000547e-04, 2.89029548e-04, 2.24826561e-04, 1.09356587e-04,
        8.88416305e-05, 4.17204974e-05,            nan,            nan,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            nan,
                   inf, 2.13460866e-03, 5.25681918e-04, 9.92588390e-04,
        3.89329777e-04, 2.54608322e-04, 2.15470792e-04, 1.21249570e-04,
        8.98361897e-06,            nan,            nan,            nan,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            nan,
                   inf,            inf, 7.53145650e-04, 8.12078784e-04,
        4.38880086e-04, 2.79701423e-04, 2.94720911e-04, 1.22199919e-04,
                   nan,            nan,            nan, 2.67573626e-05,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            nan,
                   nan,            inf, 1.00023708e-02, 3.05528593e-04,
        6.53269164e-04, 3.81193802e-04, 1.40113019e-04, 1.51322060e-04,
        1.51322060e-04,            nan, 1.12090415e-05,            nan,
        1.12090415e-05,            nan,            nan,            nan]])

# Next step is Schechter function fitting

In [ ]:
#Starting Data, we should be able to start the code from here and be solid
binedges = np.arange(40,45.25,.25)
bincenters=binedges[1:]-(binedges[1]-binedges[0])/2
counts_cor = array([[           nan,            nan,            nan,            inf,
        1.17216246e-02, 2.97925303e-03, 5.17090023e-03, 5.27232221e-03,
        2.39367967e-03, 1.67326970e-03, 0.00000000e+00, 7.73306759e-06,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            inf,
                   inf, 8.97161113e-03, 5.08561379e-03, 6.54668018e-03,
        2.70925224e-03, 1.08144954e-03, 5.94835169e-04, 2.18713174e-04,
        8.88416305e-05, 5.90016933e-05, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            nan,
                   inf, 8.80121698e-03, 2.16744208e-03, 4.65565223e-03,
        1.45673864e-03, 9.18003358e-04, 4.81807339e-04, 1.71472785e-04,
        8.98361897e-06, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            nan,
                   inf,            inf, 3.28288578e-03, 3.80898712e-03,
        1.75552035e-03, 1.00847783e-03, 7.21915847e-04, 2.44399839e-04,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 2.67573626e-05,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [           nan,            nan,            nan,            nan,
                   nan,            inf,            inf, 8.08352675e-04,
        2.53010059e-03, 1.00854400e-03, 2.80226038e-04, 1.51322060e-04,
        1.51322060e-04, 0.00000000e+00, 1.12090415e-05, 0.00000000e+00,
        1.12090415e-05, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])

errs = array([[           nan,            nan,            nan,            inf,
        3.70670317e-03, 8.98278586e-04, 1.33512070e-03, 1.31808055e-03,
        7.56947974e-04, 6.83109494e-04,            nan, 7.73306759e-06,
                   nan,            nan,            nan,            nan,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            inf,
                   inf, 1.40113026e-03, 8.04106144e-04, 1.17581847e-03,
        5.12000547e-04, 2.89029547e-04, 2.24826561e-04, 1.09356587e-04,
        8.88416305e-05, 4.17204975e-05,            nan,            nan,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            nan,
                   inf, 2.13460866e-03, 5.25681918e-04, 9.92588392e-04,
        3.89329777e-04, 2.54608321e-04, 2.15470792e-04, 1.21249569e-04,
        8.98361897e-06,            nan,            nan,            nan,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            nan,
                   inf,            inf, 7.53145650e-04, 8.12078783e-04,
        4.38880087e-04, 2.79701424e-04, 2.94720911e-04, 1.22199920e-04,
                   nan,            nan,            nan, 2.67573626e-05,
                   nan,            nan,            nan,            nan],
       [           nan,            nan,            nan,            nan,
                   nan,            inf,            inf, 3.05528593e-04,
        6.53269164e-04, 3.81193801e-04, 1.40113019e-04, 1.51322060e-04,
        1.51322060e-04,            nan, 1.12090415e-05,            nan,
        1.12090415e-05,            nan,            nan,            nan]])
bincenters, counts_cor, errs

In [ ]:
#DONT RUN AGAIN From here, we remove nan and inf values to not trip up EMCEE 
masks=[]
xs=[]
ys=[]
yerrs=[]
for num, z in enumerate(zs):
    mask = (counts_cor[num] > 0) & (counts_cor[num] < 1)
    masks.append(mask)
    xs.append(bincenters[mask])
    ys.append(counts_cor[num][mask])
    yerrs.append(errs[num][mask])


In [ ]:
#Returns all the data points from the final LFs above, including their error bars
for i in range(len(xs)):
    print(xs[i])
    print(ys[i])
    print(yerrs[i])
schechter_fits=[]

In [ ]:
#Define a schechter curve
def schechter(L, phi, Lstar, alpha):
    return phi*((L/Lstar)**alpha)*np.exp(-1*(L/Lstar))

#Define a schechter curve without a fixed alpha in logspace
def sobral_schechter_log_inputs(L, logphi,logLstar, alpha):
    return 10**logphi*((10**L/10**logLstar)**alpha) * np.exp(-1*(10**L/10**logLstar))

#Define a schechter curve with a fixed alpha in logspace
def schechter_log_inputs(L, logphi,logLstar):#, alpha):
    return np.power(10.0, logphi) * (np.power(10.0, L) / np.power(10.0, logLstar)) ** -0.75584552 * np.exp(-1 * (np.power(10.0, L) / np.power(10.0, logLstar)))


#Potential priors:
#Lstar> lowest L, maybe less than 55
#alpha is negative

In [ ]:
#same as above
def lnlike(theta, x, y, yerr):
    model = schechter_log_inputs(x, *theta)
    lnL = -0.5*np.sum((y - model) ** 2 / yerr ** 2)
  #  print(theta)
    return lnL

#same as above, with priors reflecting the data on the final LFs
def lnprior(theta):
    logphi, logLstar = theta#, alpha = theta #from the fit inputs
    
    #setting priors
    min_logphi = -10
    max_logphi = 5
    min_logLstar = 39
    max_logLstar = 55
#     max_alpha = 0
#     min_alpha = -100
    
    #applying priors
    if (min_logLstar < logLstar < max_logLstar) &(min_logphi < logphi<max_logphi):#& (min_alpha< alpha < max_alpha):
        return 0.0
    return -np.inf

#same as above
def lnprob(theta, x, y, yerr):
    lp = lnprior(theta)
    if not np.isfinite(lp):
        return -np.inf
    prob = lp + lnlike(theta, x, y, yerr)
   # print(prob)
    return prob

#use manual initial guesses, and curve fit operates the same as above
def init_emcee(emcee_x, emcee_y, emcee_yerr, diagnose = False): #diagnose will run tests
    
    #guesses made based off eyeballing the subplots...shouldn't matter much
    guess_logphi = -3
    guess_logLstar = 42
  #  guess_alpha = -1.5
    
    if diagnose == True:
        
        print('Minimization Guesses')
        print(f"log_phi: {guess_logphi}")
        print(f"log_Lstar: {guess_logLstar}")
      #  print(f"alpha: {guess_alpha}")
        print() 

    x0 = [guess_logphi, guess_logLstar]#, guess_alpha] #Initial guesses made
    bounds_low = [-np.inf, -np.inf]#, -np.inf] #maybe adjust later, I could make alpha max 0 or logLstar min 0 or 30
    bounds_high = [np.inf, np.inf]#, 0]
    
    result,_ = curve_fit(schechter_log_inputs, emcee_x, emcee_y, p0 = x0, bounds = [bounds_low, bounds_high], maxfev = 10000)  
   
    
     ########
    # Diagnostic Plotting: making sure we are getting the emission line
    ########
    if diagnose == True:
        
        print('Minimization Results')
        print(f"log_phi: {result[0]}")
        print(f"log_Lstar: {result[1]}")
       # print(f"alpha: {result[2]}")
        print()

    
    return result 

#Applies same as above, with edits made to accomodate the Schechter function, differeing jumps
def fitting_line(emcee_x, emcee_y, emcee_yerr, run = 10000,
                 diagnose = False,save_df=True, save_shec = True, 
                 file_shec = 'Emcee_Schechter_Test.txt', 
                 filename = 'Emcee_Chains_Galaxy_Test.txt'):
    
    result = init_emcee(emcee_x, emcee_y, emcee_yerr, diagnose = diagnose) #calls from above
    print(result)
    guess_logphi = result[0]
    guess_logLstar = result[1]
   # guess_alpha = result[2]
    
    #Now, create walkers to explore the parameter space
    logphi_jump = np.random.normal(loc = guess_logphi,        
                                scale = .1,      
                                size = 32).reshape(-1, 1) 
    
    logLstar_jump = np.random.normal(loc = guess_logLstar,    
                                       scale = .1,      
                                       size = 32).reshape(-1, 1)
    
#     alpha_jump = np.random.normal(loc = guess_alpha,       
#                                   scale = .05,          
#                                   size = 32).reshape(-1, 1)
  
    #################
    # Diagnostic plotting to see if the parameters were jumping to large values
    # The should be concentrated near their best fit results values
    #################
    if diagnose == True:
        print('Checking the Walker Jumps')
        fig, ax = plt.subplots(nrows = 2, ncols = 3, constrained_layout = True)
        
        ax[0, 0].hist(logphi_jump)
        ax[0, 0].set_xlabel('logphi')
        
        ax[0, 1].hist(logLstar_jump)
        ax[0, 1].set_xlabel('logLstar')
        
#         ax[0, 2].hist(alpha_jump)
#         ax[0, 2].set_xlabel('alpha')
        
     
        
   #     plt.show()
        
    #stacking along columns and creating starter walkers    
    starting_walkers = np.hstack((logphi_jump, logLstar_jump))#, alpha_jump))
    
    
    #NOTE:
    #need to change output name everytime you run otherwise it will overwrite
    ###########
    
    if save_shec == True:
        #saves the input emcee spectra
        emcee_shec_matrix = np.c_[emcee_x, emcee_y, emcee_yerr]
    
        np.savetxt(file_shec, emcee_shec_matrix)
        
    #initializing walker positions
    pos = starting_walkers
    nwalkers, ndim = pos.shape
    
    #initializing sampler
    sampler = emcee.EnsembleSampler(nwalkers, #giving emcee the walker positions
                                    ndim,     #giving it the dimension of the model(same as number of model parameters)
                                    lnprob, #giving it the log_probability function
                                    args=(emcee_x, emcee_y, emcee_yerr))#, #arguments to pass into log_probability
#                                     moves = [(emcee.moves.DEMove(), 0.5),        
#                                              (emcee.moves.DESnookerMove(), 0.5)])
    
     #running emcee
    state = sampler.run_mcmc(pos, 2000)
    sampler.reset()
    sampler.run_mcmc(state, run, progress=False)
    
    #getting values back
    flat_samples = sampler.get_chain(flat=True)
    print(flat_samples)
    LnL_chain = sampler.flatlnprobability
   # burn_in = 1000 
    
    emcee_df = pd.DataFrame()
    emcee_df['logphi'] = flat_samples[:, 0]
    emcee_df['logLstar'] = flat_samples[:, 1]
  #  emcee_df['alpha'] = flat_samples[:, 2]
    emcee_df['LnL'] = LnL_chain[:]
  #  print(emcee_df)
    emcee_df = emcee_df[np.isfinite(emcee_df.LnL.values)] #removing bad log likelihood fits
    
    
    
#     if diagnose == True:
        
#         print('Checking Parameter Posterior Distributions')
#         fig, ax = plt.subplots(nrows = 2, ncols = 2, constrained_layout = True)
        
#         emcee_df.A.hist(ax = ax[0, 0])
#         emcee_df.mu.hist(ax = ax[0, 1])
#         emcee_df.sigma.hist(ax = ax[1, 0])
#         #emcee_df.m.hist(ax = ax[1, 0])
#         emcee_df.b.hist(ax = ax[1, 1])
        
#   #      plt.show()
    
    if diagnose == True:
        xarr = np.linspace(emcee_x[0], emcee_x[-1], 100)
        
#         plt.figure()
#         plt.title('Input Emcee Spectra and Emcee Fit')
#         plt.plot(emcee_wave, emcee_spec, color = 'black', alpha = 0.5, label = 'Data')
#         plt.scatter(emcee_wave, emcee_spec, color = 'black')
#         plt.plot(xarr, comb(xarr, *emcee_df.quantile(q = 0.5).values[:-2]), label = 'Model')
#         plt.xlabel(r'Wavelength [$\mu$m]')
#         plt.ylabel('Flux')
#         plt.legend()
#   #      plt.show()
    
    
    
    
    xarr = np.linspace(emcee_x[0], emcee_x[-1], 100)

    
#     plt.figure()
#     plt.title('Input Emcee Spectra and Emcee Fit WITH 1-SIGMA SHADING')
#     plt.step(emcee_wave, emcee_spec, color = 'red', alpha = 0.5, label = 'Data', where = 'mid')
#     plt.errorbar(emcee_wave, emcee_spec, yerr = emcee_err, color = 'gray', alpha = 0.5, fmt = 'none')
    
#     l16_params = emcee_df.quantile(q = 0.16).values[:-2]
#     u84_params = emcee_df.quantile(q = 0.84).values[:-2]

#     l16_model = comb(xarr, *l16_params)#, line_center )
#     med_model = comb(xarr, *emcee_df.quantile(q = 0.5).values[:-2])#, line_center )
#     u84_model = comb(xarr, *u84_params)#, line_center )
    
#     plt.plot(xarr, med_model, label = 'Model', color = 'black')
    
#     plt.fill_between(xarr, u84_model, l16_model, color = 'dodgerblue', alpha = 0.5)
    
#     plt.xlabel(r'Wavelength [$\mu$m]')
#     plt.ylabel('Flux')
#     plt.legend()
#   #  plt.show()


#     ###########
#     #NOTE:
#     #need to also give the filename argument otherwise it will overwrite the default file
#     ###########
    if save_df == True:
        emcee_df.to_csv(filename, sep = ' ')
        
    else:
        return emcee_x, emcee_y, emcee_yerr, emcee_df


    

In [ ]:
#Running EMCEE for Schechter function fitting

check_dfs=[]
try_again=[]
for i in range(len(xs)):
    check_x, check_y, check_yerr, check_df = fitting_line(xs[i], ys[i], yerrs[i], save_df = False,diagnose = True)
    try_again.append(check_df.quantile(q=0.5))
    check_dfs.append(check_df)

In [ ]:
#save median schechter fits
# schechter_fits = [array([-2.32828593, 42.10640814, -0.1038168 ]),
#  array([-3.18413732, 42.90695785, -0.75584552]),
#  array([-2.723361,  42.452613, -0.316412]),
#  array([-5.064003,  44.649716, -0.821661]),
#  array([-4.297842, 43.467404,  -0.823897]),
#  array([-2.78, 42.87, -1.59])]

# schechter_fits = [array([-2.32828593, 42.10640814, -0.1038168 ]),
#  array([-3.18413732, 42.90695785, -0.75584552]),
#  array([-2.723361,  42.452613, -0.75584552]),
#  array([-5.064003,  44.649716, -0.75584552]),
#  array([-4.297842, 43.467404,  -0.75584552])]#,
#  #array([-2.78, 42.87, -1.59])]

#with constrained high-z alpha
schechter_fits = [array([-2.32828593, 42.10640814, -0.1038168 ]),
 array([-3.18413732, 42.90695785, -0.75584552]),
 array([-3.213253,  42.654987, -0.75584552]),
 array([-4.529986,  44.374606, -0.75584552]),
 array([-4.329837, 43.644615,  -0.75584552])]#,
 #array([-2.78, 42.87, -1.59])]

In [ ]:
#Plot all the schechter fits together, along with two older fits from Sobral et al. (2015)

# xarr = np.linspace(39.9, 50, 1000)
fit_vals=[]
plt.figure(dpi = 250)
for z in range(len(zs)):
  #  xarr = np.linspace(min(xs[z]), max(xs[z]), 1000)
    yarr=[]
    for i in xarr:
        yarr.append(sobral_schechter_log_inputs(i, schechter_fits[z][0], schechter_fits[z][1], schechter_fits[z][2]))
    fit_vals.append(yarr)
    plt.plot(xarr, yarr, label = f'z={(z+2.5)-0.5}-{(z+2.5)+0.5}', color = colors[z])
    plt.yscale("log")
    plt.ylim(1e-6, .02)
    plt.xlim(39.9, 44.25)
    plt.title("Schechter Fits with Redshift Evolution", fontsize = 16)
    plt.xlabel("Log(L) (erg/s)", fontsize = 13)
    plt.ylabel("Log(ϕ) (Mpc$^{-3}$dlogL$^{-1}$)", fontsize = 13)
    #plt.legend()
    
#Add Sobral fits
# sobral_fits = [-2.78, 42.87, -1.59]
# sobral_fits2 = [-2.23, 42.23, -.9]
# sobral=[]
# sobral2=[]
# for i in xarr:
#     sobral.append(sobral_schechter_log_inputs(i, sobral_fits[0], sobral_fits[1], sobral_fits[2]))
#     sobral2.append(sobral_schechter_log_inputs(i, sobral_fits2[0], sobral_fits2[1], sobral_fits2[2]))
# plt.plot(xarr, sobral, label = "z=2.23 with H-α (Sobral)", color = 'black', linestyle = "--")
# plt.plot(xarr, sobral2, label = "z=2.20 with [OII] (Sobral)", color = "grey", linestyle = "--")
plt.legend()
plt.show()

In [ ]:
#loop my integral for the last 1000 (3200) runs to get error bars
#I can't take the 1-sigma from each parameter. I instead have to integrate everything, then take the 1-sigma of the integrals and see what the error bars are there
#check_dfs[0][288000:]
from scipy.integrate import simps
xarr = np.linspace(39.9, 50, 1000)
low_bound=41.25
all_range_integrals=[]
for z in range(len(check_dfs)):
    range_integrals=[]
    df_subset = check_dfs[z][288000:][['logphi', 'logLstar']].to_numpy()#, 'alpha']].to_numpy()
    for logphi, logLstar in df_subset:#, alpha in df_subset:
   # for index, row in check_dfs[z][-10:].iterrows(): #iterating through rows since pandas is weird with columns indices
    #    print(row['logphi'], row['logLstar'], row['alpha'])

        #creating schechter function
        yarr=[]
        for i in xarr:
            yarr.append(sobral_schechter_log_inputs(i, logphi, logLstar, -0.75584552)) 
        yarr=np.array(yarr)

        #applying integration bounds
        slicex = xarr[xarr>low_bound]
        slicey = ((10**xarr)*yarr)[xarr>low_bound]

        #Take integral
        simps_integral = simps(slicey, slicex)
        range_integrals.append(simps_integral)

    #print(range_integrals)
    all_range_integrals.append(range_integrals)
    print(len(range_integrals))

print(len(all_range_integrals))
all_range_integrals


In [ ]:
#Sort the EMCEE result fits in order
all_range_integrals[0].sort()
all_range_integrals[1].sort()
all_range_integrals[2].sort()
all_range_integrals[3].sort()

In [ ]:
#print the integrated values under the curve
integrals = [2.323616441620799e+39,
 4.834384620999347e+39,
 2.2440751047323246e+39,
 9.137278958358004e+39,
 2.375907580093392e+39]

In [ ]:
#Print the 1-sigma range of EMCEE fit results
(all_range_integrals[1][5120], all_range_integrals[1][26880], all_range_integrals[1][16000])

In [ ]:
#Now i need to get the 16-84% of the INTEGRAL for each, and then *7.9e-42 and make those the error bounds of my SFRD plot
upper_bars = []
lower_bars = []
for i in range(len(zs[1:])):
    upper_bars.append(all_range_integrals[i][26880])
    lower_bars.append(all_range_integrals[i][5120])
    
upper_bars = np.array(upper_bars)
lower_bars = np.array(lower_bars)

upper_bars = upper_bars*7.9e-42
lower_bars = lower_bars*7.9e-42

upper_bars, lower_bars

In [ ]:
#integrate the L*schechter function (w/out bounds or with the low and high of my data as bounds
#then take that number, multiply it by the kennicut factor (or rather, the sobral version)
low_bound=41.25

integrals = []
for i in range(len(fit_vals)):
    fit_vals[i] = np.array(fit_vals[i]) #make into array
  #  print(fits[i])
    #Only integrate from the ranges we actually have points
#     idx_a = np.searchsorted(xarr, min(xs[i]))
#     idx_b = np.searchsorted(xarr, max(xs[i]))
    slicex = xarr[xarr>low_bound]
    slicey = ((10**xarr)*fit_vals[i])[xarr>low_bound]
    
    #Take integral
    simps_integral = simps(slicey, slicex)
    integrals.append(simps_integral)
    

sobral = np.array(sobral)
#only integrate from the ranges we actually have points
# idx_a = np.searchsorted(xarr, low_bound)#-.075)
# idx_b = np.searchsorted(xarr, 43.3)#+.05)
sobral_slicex = xarr[xarr>low_bound]
sobral_slicey = ((10**xarr)*sobral)[xarr>low_bound]
simps_sobral = simps(sobral_slicey, sobral_slicex) #simpson integrator good for even x-

sobral2 = np.array(sobral2)
sobral2_slicex = xarr[xarr>low_bound]
sobral2_slicey = ((10**xarr)*sobral2)[xarr>low_bound]
simps_sobral2 = simps(sobral2_slicey, sobral2_slicex)

print(simps_sobral*7.9e-42)
print(simps_sobral2*7.9e-42)
np.array(integrals)*7.9e-42

In [ ]:
#84th and 16th percentiles (from old)
# upper_bars, lower_bars = (array([0.02152935, 0.05519748, 0.02293221, 0.21461162, 0.03146043]),
#  array([0.01401819, 0.03341457, 0.01657101, 0.02966494, 0.01269627]))

upper_bars, lower_bars = (array([0.02152935, 6.48246522e-02, 2.44229573e-02, 3.81869649e+02, 1.66665305e+01]),
 array([0.01401819, 2.20500585e-02, 1.21631623e-02, 9.21838913e-05, 1.03448005e-04])) #new bars! The first zbin is the same

upper_bars = upper_bars-np.array(integrals)*7.9e-42 #take the difference between the edges and the median
lower_bars = np.array(integrals)*7.9e-42 -lower_bars
print(upper_bars, lower_bars)
xerr_range = [.5,.5,.5,.5,.5]

sobral_yerr = [.15]
sobral_lower_xerr = [.53] #2.23 h-alpha
sobral_higher_xerr = [.57]

sobral2_yerr = [.02]
sobral2_lower_xerr = [.5] #2.20 [OII]
sobral2_higher_xerr = [.6]

In [ ]:
#Apply the Kennicutt conversion factor to convert into units of SFRD
colors5 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
sobral_x = [2.23] #1.7-2.8
sobral2_x = [2.2]
plt.figure(dpi = 250)
plt.errorbar(sobral2_x, simps_sobral2*7.9e-42, xerr = [sobral2_lower_xerr, sobral2_higher_xerr], yerr = sobral2_yerr, markersize = 5, ecolor = "grey", fmt = "none")
plt.scatter(sobral2_x, simps_sobral2*7.9e-42, s=25,color = "grey", label = "(Sobral) [OII]")
plt.errorbar(sobral_x, simps_sobral*7.9e-42, xerr = [sobral_lower_xerr, sobral_higher_xerr], yerr=sobral_yerr, markersize = 5, ecolor="black", fmt = "none")
plt.scatter(sobral_x, simps_sobral*7.9e-42, color = "black", s = 25, label = "Sobral (H-α)")
plt.errorbar(zs, np.array(integrals)*7.9e-42, xerr = xerr_range,yerr = [lower_bars, upper_bars],  markersize = 10, ecolor = colors5, fmt = 'none')
plt.scatter(zs, np.array(integrals)*7.9e-42, s=25, color = colors5)#, label = "This Study")
plt.title("SFR Density of the Universe", fontsize = 16)
plt.xlabel("Redshift", fontsize = 13)
plt.ylabel("Log(ρSFR) (M$_{☉}$yr$^{-1}$Mpc$^{-3}$)", fontsize = 13)
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
#Use cosmology parameters from astropy to include an uppwer axis with the age of the Universe

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import fsolve

# Cosmology parameters
H0 = 70  # Hubble constant in km/s/Mpc
Omega_m = 0.3  # Matter density
Omega_L = 0.7  # Dark energy density
H0_sec = H0 / (3.086e19)  # Convert H0 to 1/s

# Define Hubble function
def H(z):
    return H0_sec * np.sqrt(Omega_m * (1 + z)**3 + Omega_L)

# Function to calculate age of the universe at redshift z
def age_universe(z):
    if np.isscalar(z):  # Handle scalar input
        integral, _ = quad(lambda zp: 1 / ((1 + zp) * H(zp)), z, np.inf)
        return integral / (60 * 60 * 24 * 365 * 1e9)  # Convert to Gyr
    else:  # Handle array input
        return np.array([age_universe(zi) for zi in z])

# Functions for secondary x-axis transformation
def redshift_to_age(z):
    return age_universe(z)  # Now supports both scalars and arrays

def age_to_redshift(age):
    if np.isscalar(age):  # Ensure fsolve works with scalars
        return fsolve(lambda z: redshift_to_age(z) - age, x0=2)[0]
    else:
        return np.array([age_to_redshift(a) for a in age])  # Handle array input



# Create figure
fig, ax1 = plt.subplots(figsize=(8, 6), dpi=250)

# Plot SFRD data
ax1.errorbar(sobral2_x, simps_sobral2*7.9e-42, xerr = [sobral2_lower_xerr, sobral2_higher_xerr], yerr = sobral2_yerr, markersize = 5, ecolor = "grey", fmt = "none")
ax1.scatter(sobral2_x, simps_sobral2*7.9e-42, color = "grey", label = "(Sobral) [OII]", s = 50)
ax1.errorbar(sobral_x, simps_sobral*7.9e-42, xerr = [sobral_lower_xerr, sobral_higher_xerr], yerr=sobral_yerr, markersize = 5, ecolor="black", fmt = "none")
ax1.scatter(sobral_x, simps_sobral*7.9e-42, color = "black", s = 50, label = "Sobral (H-α)")
ax1.errorbar(zs, np.array(integrals)*7.9e-42, xerr = xerr_range,yerr = [lower_bars, upper_bars],  markersize = 10, ecolor = colors5, fmt = 'none')
ax1.scatter(zs, np.array(integrals)*7.9e-42, s=50, color = colors5, label = "This Study")
ax1.set_xlabel("Redshift", fontsize=13)
ax1.set_ylabel(r"Log(ρSFR) (M$_{\odot}$ yr$^{-1}$ Mpc$^{-3}$)", fontsize=13)
ax1.set_yscale("log")
ax1.set_ylim(1e-3,1)
ax1.set_title("SFR Density of the Universe", fontsize=18)


ax2 = ax1.secondary_xaxis("top", functions=(redshift_to_age, age_to_redshift))
ax2.set_xlabel("Age of the Universe (Gyr)", fontsize=13)

# Get tick positions from the primary x-axis
redshift_ticks = ax1.get_xticks()  

# Compute corresponding age values
age_ticks = redshift_to_age(redshift_ticks)  

# Set the ticks on the top axis
ax2.set_xticks(age_ticks)  
ax2.set_xticklabels([f"{age:.1f}" for age in age_ticks])  # Format as Gyr

plt.legend()
plt.show()



In [ ]:
#Old attempt with less constrained alpha values on the Schechter fits...perhaps for further exploration another day

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import fsolve

# Cosmology parameters
H0 = 70  # Hubble constant in km/s/Mpc
Omega_m = 0.3  # Matter density
Omega_L = 0.7  # Dark energy density
H0_sec = H0 / (3.086e19)  # Convert H0 to 1/s

# Define Hubble function
def H(z):
    return H0_sec * np.sqrt(Omega_m * (1 + z)**3 + Omega_L)

# Function to calculate age of the universe at redshift z
def age_universe(z):
    if np.isscalar(z):  # Handle scalar input
        integral, _ = quad(lambda zp: 1 / ((1 + zp) * H(zp)), z, np.inf)
        return integral / (60 * 60 * 24 * 365 * 1e9)  # Convert to Gyr
    else:  # Handle array input
        return np.array([age_universe(zi) for zi in z])

# Functions for secondary x-axis transformation
def redshift_to_age(z):
    return age_universe(z)  # Now supports both scalars and arrays

def age_to_redshift(age):
    if np.isscalar(age):  # Ensure fsolve works with scalars
        return fsolve(lambda z: redshift_to_age(z) - age, x0=2)[0]
    else:
        return np.array([age_to_redshift(a) for a in age])  # Handle array input



# Create figure
fig, ax1 = plt.subplots(figsize=(8, 6), dpi=250)

# Plot SFRD data
ax1.scatter(sobral_x, simps_sobral*7.9e-42, color="black", s=100, label="Sobral et al.")
ax1.scatter(zs, np.array(integrals) * 7.9e-42, s=100)
ax1.set_xlabel("Redshift", fontsize=13)
ax1.set_ylabel(r"Log(ρSFR) (M$_{\odot}$ yr$^{-1}$ Mpc$^{-3}$)", fontsize=13)
ax1.set_yscale("log")
ax1.set_title("SFR Density of the Universe", fontsize=18)

ax2 = ax1.secondary_xaxis("top", functions=(redshift_to_age, age_to_redshift))
ax2.set_xlabel("Age of the Universe (Gyr)", fontsize=13)

# Get tick positions from the primary x-axis
redshift_ticks = ax1.get_xticks()  

# Compute corresponding age values
age_ticks = redshift_to_age(redshift_ticks)  

# Set the ticks on the top axis
ax2.set_xticks(age_ticks)  
ax2.set_xticklabels([f"{age:.1f}" for age in age_ticks])  # Format as Gyr

plt.legend()
plt.show()

